# Korea FCFF Batch Valuation **v1** (base: v8 모형)

KRX universe 전체(또는 지정 범위)를 대상으로 FCFF DCF 를 **대량 배치 실행**하고 결과를 **DB(`korea_fcff_dcf_valuation_v8`)** 에 저장하는 노트북.

- 모형 로직·DB 저장 방식·저장 내용은 `korea_fcff_dcf_valuation_v8` 과 동일 (US v13 Validity 가드 포함)
- **모든 입력 변수는 Cell 2 한 곳**에 집중 — Cell 2 만 수정하고 나머지는 위→아래 순서대로 실행
- 단일 종목 진단/시각화/Excel 출력 셀은 제거 → 배치 전용 (개별 평가는 `korea_fcff_individual_valuation_v1` 사용)

## Cell 1 · 경로 자동 감지 (노트북/데스크탑 공용)

In [1]:
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────
#  프로젝트 루트(= DATA 폴더의 부모)를 sys.path 에 등록.
#  → 노트북/데스크탑 어느 PC 에서 실행해도
#     from DATA.config import ...
#     from DATA.us_target_ticker_list_screened_20260411 import ...
#     처럼 DATA.* 모듈이 동일하게 import 됩니다.
#  ▸ 노트북   : C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast\DATA
#  ▸ 데스크탑 : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA
# ─────────────────────────────────────────────────────────────
_CANDIDATE_ROOTS = [
    r"C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast",
    r"C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy",
]

def _setup_path() -> str:
    try:
        start = Path(__file__).resolve().parent
    except NameError:
        start = Path.cwd()
    for p in [start] + list(start.parents):
        if (p / "DATA").is_dir():
            root = str(p)
            if root not in sys.path:
                sys.path.insert(0, root)
            print(f"[PATH] root 자동 감지 : {root}")
            return root
    for cand in _CANDIDATE_ROOTS:
        if os.path.isdir(cand) and os.path.isdir(os.path.join(cand, "DATA")):
            if cand not in sys.path:
                sys.path.insert(0, cand)
            print(f"[PATH] root 후보 경로 : {cand}")
            return cand
    raise EnvironmentError(
        "DATA 폴더를 찾을 수 없습니다. _CANDIDATE_ROOTS 를 환경에 맞게 수정하세요."
    )

_ROOT = _setup_path()
print(f"[확인] 프로젝트 루트 : {_ROOT}")
print(f"[확인] DATA 경로    : {os.path.join(_ROOT, 'DATA')}")


[PATH] root 자동 감지 : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy
[확인] 프로젝트 루트 : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy
[확인] DATA 경로    : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA


## Cell 2 · ★ 입력 변수 (여기만 수정하세요)

- `TICKER_START / TICKER_END / SKIP_DONE` : 배치 범위·이어하기
- `RUN_TICKERS_OVERRIDE` : 특정 종목만 배치 실행할 때
- `TOP_N / RANK_DATES_INPUT` : 배치 후 랭킹 조회 설정
- 이하 모델 파라미터·v8 가드 상수 (기본값 = v8 과 동일)

In [2]:
# ═══════════════════════════════════════════════════════════════
#  ★★★ 입력 변수 셀 — 이 노트북의 모든 사용자 입력은 여기 한 곳 ★★★
#  (아래 셀들은 수정할 필요가 없습니다)
# ═══════════════════════════════════════════════════════════════

# ─────────────────────────────────────────────────────────────
#  ① 배치 실행 범위
# ─────────────────────────────────────────────────────────────
TICKER_START = 0          # universe 리스트에서 시작 인덱스 (0 = 처음부터)
TICKER_END   = 9999       # universe 리스트에서 끝 인덱스 (9999 = 끝까지)
SKIP_DONE    = False      # True → 체크포인트(done_tickers.txt)에 기록된 종목 건너뜀
                          #        (중단된 배치를 이어서 실행할 때 True 로)

RUN_TICKERS_OVERRIDE = [] # 특정 종목만 배치 실행하고 싶을 때 사용.
                          # 비워두면([]) universe 전체 슬라이스 실행.
                          # 예: ["A065450", "A005930"]

# ─────────────────────────────────────────────────────────────
#  ② Universe 필터
# ─────────────────────────────────────────────────────────────
MIN_REVENUE_QUARTERS = 24 # universe 편입 조건: 최소 보유 매출 분기 수 (24 = 6년)

# ─────────────────────────────────────────────────────────────
#  ③ 랭킹 조회 (배치 완료 후 마지막 셀에서 사용)
# ─────────────────────────────────────────────────────────────
TOP_N            = 600    # upside 상위 추출 종목 수
RANK_DATES_INPUT = []     # 랭킹에 사용할 측정일 리스트.
                          # []  → 가장 최근 측정일 1일 자동 사용
                          # 예: ["2026-07-07", "2026-07-06"]

# ─────────────────────────────────────────────────────────────
#  ④ DB 테이블 (변경 시에만 수정)
# ─────────────────────────────────────────────────────────────
TABLE_FS         = "korea_fs_data_from_DG"            # 재무제표 원본 (DataGuide)
TABLE_FORECAST   = "korea_revenue_forecast_result"    # 매출 예측 결과
TABLE_MARKETCAP  = "ks_listed_company_daily_marketcap"# 일별 시가총액
TABLE_PRICE      = "KSE_Price"                        # 일별 주가
TABLE_RESULT     = "korea_fcff_dcf_valuation_v8"      # ★ v8 결과 저장 테이블 (버전별 분리)
TABLE_QUALITY    = "korea_valuation_quality_log"      # 데이터 품질 로그

# ─────────────────────────────────────────────────────────────
#  ⑤ 모델 파라미터
# ─────────────────────────────────────────────────────────────
FORECAST_HORIZON     = 8            # Phase 1 예측 분기 수 (8분기 = 2년)
MIN_HISTORY          = 16           # OPM 회귀에 필요한 최소 분기 수
WINSORIZE_LIMITS     = (0.05, 0.95) # 비율 계수 추정 시 winsorize 범위
GDP_GROWTH           = 0.04         # 한국 장기 GDP 성장률 추정 (terminal g 상한)

# Phase 2 성장률(g) 허용 범위 (US v9.2 패치)
G_FLOOR_MIN          = -0.20        # 음수 g₀ 하한 (사이클릭 침체 -20% 까지 허용)
G_CEIL               = 1.00         # ★ v8: g 상한 (US v13 정렬, 구 0.50)
                                    #   적응형 ρ 상한·TV 배수 상한이 받쳐주므로 상향 안전

# OLS fallback 기준 (비율 계수 회귀가 이 기준 미달이면 median fallback)
OLS_MIN_R2           = 0.30         # 회귀 채택 최소 R²
OLS_MIN_SAMPLES      = 16           # 회귀 채택 최소 표본 수

# ─────────────────────────────────────────────────────────────
#  ⑥ WACC / ERP
# ─────────────────────────────────────────────────────────────
ERP_METHOD           = "damodaran_floor"  # 'damodaran_floor' | 'kospi_geo_10y' | 'kospi_geo_no_floor'
DAMODARAN_ERP_KR     = 0.07               # Damodaran 한국 ERP
GEO_FLOOR            = 0.07               # KOSPI geo 사용 시 E(Rm) floor
RF_FALLBACK          = 0.035              # BOK API 실패 시 fallback 무위험이자율
RD_DEFAULT           = 0.045              # 부채비용(Rd) 산출 실패 시 기본값
WACC_FLOOR           = 0.05               # WACC 절대 하한 (v8 은 max(이 값, Rf+1%) 사용)
WACC_CAP             = 0.20               # WACC 상한

# ─────────────────────────────────────────────────────────────
#  ⑦ ★ v8 Validity 가드 (미국 v13 이식, 통화가드 제외)
# ─────────────────────────────────────────────────────────────
V8_RD_FLOOR_RF      = True     # ② Rd 하한 = Rf (forward-looking 부채비용)
V8_WACC_FLOOR_ABS   = 0.05     # ② WACC 절대 하한 → max(이 값, Rf+1%)
V8_MIN_TV_SPREAD    = 0.02     # ③ WACC − g_term 최소 스프레드 (구 0.01)
V8_TERMINAL_RF_CAP  = True     # ③ g_term ≤ min(GDP, Rf)  (Damodaran)
V8_TV_FCFF_MULT_CAP = 35.0     # ③ TV / 마지막 연간 FCFF 배수 상한
V8_SANITY_GUARD     = True     # ④ 지분가치/시총 괴리 검사
V8_SANITY_MC_RATIO  = 30.0     # ④ 허용 배율 (초과 시 평가 제외)
V8_RHO_ADAPTIVE_CAP = True     # ⑤ g0 기반 ρ 적응형 상한

# ─────────────────────────────────────────────────────────────
#  ⑧ Net Debt / NWC 구성 항목 (BS 컬럼 키)
# ─────────────────────────────────────────────────────────────
DEBT_KEYS = ["short_term_debt", "current_lt_debt", "bonds",
             "long_term_debt", "lease_liab"]           # 총부채 합산 항목
CASH_KEYS = ["cash", "short_term_invest"]              # 현금성 합산 항목

# ★ v7 비현금 영업운전자본 (Damodaran 표준)
#   NWC = (매출채권+재고자산+선급비용) − (매입채무+미지급비용+미지급금+선수금+계약부채)
USE_OPERATING_NWC = True   # False → 구 v6 proxy 로 회귀 (A/B 비교용)

# FCFF_RIM_DATA_2026_1Q 에 추가된 BS 영업운전자본 항목 (참조용)
#   (helper 의 DG_ITEM_CODES 매핑 + DB 적재가 선행되어야 함)
V7_NEW_ITEM_CODES = {
    "M000902006": "payables",              # 매입채무
    "M001122290": "accrued_expenses",      # 미지급비용
    "M001122140": "other_payables",        # 미지급금
    "M001122580": "advances_received",     # 선수금
    "M001122561": "contract_liabilities",  # 계약부채
    "M001113970": "prepaid_expenses",      # 선급비용 (영업유동자산)
}

# ─────────────────────────────────────────────────────────────
#  ⑨ 단위 환산 / 체크포인트 (수정 불필요)
# ─────────────────────────────────────────────────────────────
MARKETCAP_UNIT_MULTIPLIER = 1_000_000  # 시가총액: 백만원 → 원
FS_UNIT_MULTIPLIER        = 1_000      # 재무제표: 천원 → 원

CHECKPOINT_DIR = "_korea_fcff_checkpoint"  # 진행 상황 기록 폴더
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
DONE_PATH = os.path.join(CHECKPOINT_DIR, "done_tickers.txt")    # 완료 종목 기록
FAIL_PATH = os.path.join(CHECKPOINT_DIR, "failed_tickers.txt")  # 실패 종목 기록

print("[OK] 입력 변수 설정 완료")
print(f"  배치 범위        : [{TICKER_START}:{TICKER_END}]  SKIP_DONE={SKIP_DONE}"
      f"  OVERRIDE={len(RUN_TICKERS_OVERRIDE)}개")
print(f"  결과 테이블      : {TABLE_RESULT}")
print(f"  Phase2 g 범위    : [{G_FLOOR_MIN:.0%}, {G_CEIL:.0%}]  GDP={GDP_GROWTH:.0%}")
print(f"  v8 가드          : Rd≥Rf={V8_RD_FLOOR_RF}  TV스프레드≥{V8_MIN_TV_SPREAD:.0%}"
      f"  TV배수≤{V8_TV_FCFF_MULT_CAP:.0f}×  Sanity±{V8_SANITY_MC_RATIO:.0f}배")


[OK] 입력 변수 설정 완료
  배치 범위        : [0:9999]  SKIP_DONE=False  OVERRIDE=0개
  결과 테이블      : korea_fcff_dcf_valuation_v8
  Phase2 g 범위    : [-20%, 100%]  GDP=4%
  v8 가드          : Rd≥Rf=True  TV스프레드≥2%  TV배수≤35×  Sanity±30배


## Cell 3 · Import & DB 연결

In [3]:
# ═══════════════════════════════════════════════════════════════
#  Import & DB 연결 — 사용자 입력 없음 (실행만 하면 됩니다)
# ═══════════════════════════════════════════════════════════════

# ── 표준 라이브러리 ──────────────────────────────────────────────
import gc, math, time, traceback, warnings
from datetime import datetime, date, timedelta
from typing import Optional, Dict, Any, List, Tuple

# ── 외부 라이브러리 ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import pymysql
from scipy import stats
from IPython.display import display
import matplotlib
matplotlib.rcParams["axes.unicode_minus"] = False
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sqlalchemy import text

# ── 내부 모듈 (Cell 1 경로 자동 감지 덕분에 두 PC 모두 동일하게 import) ──
from DATA.config import get_db_info, get_engine
from DATA.KEYS import KEYS
from DATA.korea_valuation_helpers import (
    setup_project_path, to_dg_ticker, to_price_ticker, get_pymysql_conn,
    DG_ITEM_CODES, get_item_code,
    load_korea_financials_wide, load_korea_revenue_forecast,
    load_korea_marketcap_latest, load_korea_price_series, load_current_price,
    load_kospi_series, get_risk_free_rate, compute_beta_10y,
    estimate_market_return, get_universe_with_min_history,
    DataQualityReport, save_quality_report_to_db, get_evaluation_history,
)
from DATA.universal_ts_forecast_function_v2 import (
    forecast_sarima, forecast_ets, forecast_theta,
    infer_freq_alias, seasonal_periods_from_freq, clear_memory,
)
# (필요 시) US 스크리닝 티커 리스트도 동일 방식으로 import 가능:
# from DATA.us_target_ticker_list_screened_20260411 import TARGET_TICKER_LIST

def log(tag: str, msg: str):
    ts = datetime.now().strftime("%H:%M:%S")
    print(f"[{ts}][{tag}] {msg}", flush=True)

# ── DB 연결 ──────────────────────────────────────────────────────
db_info = get_db_info()
engine  = get_engine(db_info)

try:
    with engine.connect() as c:
        c.execute(text("SELECT 1"))
    log("DB", f"연결 성공 host={db_info.get('host')} port={db_info.get('port')}")
except Exception as e:
    log("DB", f"연결 실패: {e}")

print(f"[OK] Import 완료")
print(f"[설정] FORECAST_HORIZON={FORECAST_HORIZON}Q  ERP_METHOD={ERP_METHOD}")
print(f"[설정] WACC 범위 [{WACC_FLOOR:.0%}, {WACC_CAP:.0%}]  Phase2 g 범위 [{G_FLOOR_MIN:.0%}, {G_CEIL:.0%}]")


[15:44:04][DB] 연결 성공 host=192.168.0.230 port=3307
[OK] Import 완료
[설정] FORECAST_HORIZON=8Q  ERP_METHOD=damodaran_floor
[설정] WACC 범위 [5%, 20%]  Phase2 g 범위 [-20%, 100%]


## Cell 4 · 결과 테이블 초기화 & Universe 조회

- PK = (ticker, date, quarter) → 같은 평가일+분기는 갱신, 평가일이 다르면 별도 row (시계열 추적)
- Universe: 매출 ≥ `MIN_REVENUE_QUARTERS` 분기 보유 종목

In [4]:
# ═══════════════════════════════════════════════════════════════
#  결과 테이블 초기화 & Universe 조회 — 사용자 입력 없음
#  - PK = (ticker, date, quarter) → 같은 평가일+분기는 갱신, 날짜 다르면 시계열 누적
# ═══════════════════════════════════════════════════════════════

# ── 결과 테이블 생성 (없으면) ─────────────────────────────────
CREATE_SQL = f'''
CREATE TABLE IF NOT EXISTS `{TABLE_RESULT}` (
  `id`                BIGINT      NOT NULL AUTO_INCREMENT,
  `date`              DATE        NOT NULL COMMENT '평가 실행일',
  `ticker`            VARCHAR(20) NOT NULL COMMENT 'A005930 형식',
  `quarter`           VARCHAR(10)          COMMENT 'e.g. 2026Q1',
  `sales_actual`      DOUBLE               COMMENT '과거 Sales (원)',
  `sales_forecast`    DOUBLE               COMMENT '예측 Sales (원)',
  `opm_forecast`      DOUBLE,
  `ebit`              DOUBLE,
  `tax_rate`          DOUBLE,
  `nopat`             DOUBLE,
  `da`                DOUBLE,
  `capex`             DOUBLE,
  `nwc`               DOUBLE,
  `delta_nwc`         DOUBLE,
  `fcff`              DOUBLE,
  `roic`              DOUBLE,
  `reinvestment_rate` DOUBLE,
  `g_terminal`        DOUBLE,
  `discount_rate`     DOUBLE COMMENT 'WACC',
  `wacc_re`           DOUBLE COMMENT 'Cost of Equity',
  `wacc_rd`           DOUBLE COMMENT 'Cost of Debt',
  `beta_raw`          DOUBLE,
  `beta_blume`        DOUBLE,
  `enterprise_value`  DOUBLE,
  `net_debt`          DOUBLE,
  `equity_value`      DOUBLE,
  `shares`            DOUBLE,
  `target_price`      DOUBLE,
  `current_price`     DOUBLE,
  `upside_pct`        DOUBLE,
  `moat_label`        VARCHAR(30),
  `eva_spread`        DOUBLE,
  `moat_rho`          DOUBLE      COMMENT '★v8 Phase2 지속성 ρ (적응형 상한 반영)',
  `rho_cap_applied`   TINYINT     COMMENT '★v8 g0 기반 ρ 적응형 상한 적용 여부',
  `tv_weight_pct`     DOUBLE      COMMENT '★v8 TV 가 EV 에서 차지하는 비중(%)',
  `tv_fcff_mult`      DOUBLE      COMMENT '★v8 TV / 마지막 연간 FCFF 배수',
  `tv_capped`         TINYINT     COMMENT '★v8 TV 배수 상한 적용 여부',
  `mc_ratio`          DOUBLE      COMMENT '★v8 지분가치 / 실제 시총 배율 (Sanity)',
  `revenue_quarters`  INT COMMENT '사용된 매출 분기 수',
  `forecast_model`    VARCHAR(20) COMMENT 'Sales 예측 모델명',
  `forecast_date`     DATE COMMENT '매출 예측 실행일',
  `nwc_method`        VARCHAR(20) COMMENT 'operating_v7 | legacy_proxy',
  `nwc_to_sales`      DOUBLE      COMMENT 'gamma = NWC/Sales 계수',
  `op_current_assets` DOUBLE      COMMENT '비현금 영업유동자산(최근, 원)',
  `op_current_liab`   DOUBLE      COMMENT '비차입 영업유동부채(최근, 원)',
  `created_at`        DATETIME    DEFAULT CURRENT_TIMESTAMP,
  PRIMARY KEY (`id`),
  UNIQUE KEY uq_main (`ticker`, `date`, `quarter`),
  INDEX idx_ticker (`ticker`),
  INDEX idx_date   (`date`)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4
'''

conn = get_pymysql_conn(db_info)
try:
    with conn.cursor() as cur:
        cur.execute(CREATE_SQL)
    conn.commit()
    log("DB", f"테이블 준비 완료: {TABLE_RESULT}")
finally:
    conn.close()

# ★ v7: 기존 테이블에 신규 컬럼 보강 (이미 있으면 무시)
_ALTER_COLS = [
    ("nwc_method",        "VARCHAR(20)"),
    ("nwc_to_sales",      "DOUBLE"),
    ("op_current_assets", "DOUBLE"),
    ("op_current_liab",   "DOUBLE"),
    ("moat_rho",          "DOUBLE"),     # ★ v8
    ("rho_cap_applied",   "TINYINT"),    # ★ v8
    ("tv_weight_pct",     "DOUBLE"),     # ★ v8
    ("tv_fcff_mult",      "DOUBLE"),     # ★ v8
    ("tv_capped",         "TINYINT"),    # ★ v8
    ("mc_ratio",          "DOUBLE"),     # ★ v8
]
conn = get_pymysql_conn(db_info)
try:
    with conn.cursor() as cur:
        for _c, _t in _ALTER_COLS:
            try:
                cur.execute(f"ALTER TABLE `{TABLE_RESULT}` ADD COLUMN `{_c}` {_t}")
            except Exception:
                pass  # 이미 존재
    conn.commit()
finally:
    conn.close()

# Universe 미리 조회
universe_df = get_universe_with_min_history(
    db_info, min_quarters=MIN_REVENUE_QUARTERS, require_consecutive=False
)
KOREA_TICKER_LIST = universe_df["ticker"].tolist()
print(f"\n[Universe] 매출 ≥{MIN_REVENUE_QUARTERS}분기 종목: {len(KOREA_TICKER_LIST):,}개")
print(f"  최장 보유: {universe_df['n_quarters'].max()}분기")
print(f"  평균 결측 분기: {universe_df['gap_count'].mean():.1f}")
display(universe_df.head(10))


[15:44:04][DB] 테이블 준비 완료: korea_fcff_dcf_valuation_v8

[Universe] 매출 ≥24분기 종목: 1,289개
  최장 보유: 67분기
  평균 결측 분기: 0.1


,ticker,n_quarters,first_date,last_date,gap_count
0,A017800,67,2009-12-30,2026-06-30,0
1,A017900,67,2009-12-30,2026-06-30,0
2,A018880,67,2009-12-30,2026-06-30,0
3,A020120,67,2009-12-30,2026-06-30,0
4,A020760,67,2009-12-30,2026-06-30,0
5,A023530,67,2009-12-30,2026-06-30,0
6,A023810,67,2009-12-30,2026-06-30,0
7,A025000,67,2009-12-30,2026-06-30,0
8,A025560,67,2009-12-30,2026-06-30,0
9,A025820,67,2009-12-30,2026-06-30,0


## Cell 5 · 시장 파라미터 (Rf, KOSPI, E(Rm)) — 1회 계산 후 캐싱

배치 시작 전에 계산해 모든 종목이 공유:
- `RF` : BOK API → 10Y 국고채 (실시간)
- `KOSPI_PX` : KOSPI 종합지수 (베타·E(Rm) 계산용)
- `E_RM`, `ERP` : Damodaran 또는 KOSPI geo 10y

In [5]:
# 1. Risk-free rate (BOK API)
RF, RF_SOURCE = get_risk_free_rate(KEYS["BOK"], fallback_rate=RF_FALLBACK)
log("MKT", f"Rf = {RF:.4%}  (source: {RF_SOURCE})")

# 2. KOSPI 시계열 (12년치 → 베타·E(Rm) 모두 사용)
KOSPI_PX = load_kospi_series(start_date=(datetime.today() - timedelta(days=365*12)).strftime("%Y-%m-%d"))
log("MKT", f"KOSPI {len(KOSPI_PX):,}거래일  ({KOSPI_PX.index.min().date()} ~ {KOSPI_PX.index.max().date()})")
log("MKT", f"  현재 KOSPI = {KOSPI_PX.iloc[-1]:,.2f}")

# 3. E(Rm) 추정
mkt = estimate_market_return(
    method=ERP_METHOD, rf=RF, kospi_series=KOSPI_PX,
    years=10, damodaran_erp_kr=DAMODARAN_ERP_KR, geo_floor=GEO_FLOOR,
)
E_RM = mkt["e_rm"]
ERP  = mkt["erp"]
log("MKT", f"E(Rm) = {E_RM:.4%}  ERP = {ERP:.4%}")
log("MKT", f"  → {mkt['note']}")
if mkt["kospi_geo"] is not None:
    log("MKT", f"  (참고) KOSPI 10년 기하평균 = {mkt['kospi_geo']:.4%}")

print()
print("=" * 60)
print(f"  Rf  = {RF:>7.3%}    (BOK 10Y 국고채)")
print(f"  ERP = {ERP:>7.3%}    (방식: {ERP_METHOD})")
print(f"  E(Rm) = {E_RM:>7.3%}")
print("=" * 60)


[15:44:12][MKT] Rf = 4.2880%  (source: BOK_2026-08-26)
[KOSPI] try source=fdr (2014-08-30 ~ 2026-08-27) ... FAIL (FDR 실패 (재시도 3회): LOGOUT)
[KOSPI] try source=pykrx (2014-08-30 ~ 2026-08-27) ... FAIL ('지수명')
[KOSPI] try source=yfinance (2014-08-30 ~ 2026-08-27) ... OK  2,936거래일
[15:44:26][MKT] KOSPI 2,936거래일  (2014-09-01 ~ 2026-08-26)
[15:44:26][MKT]   현재 KOSPI = 6,808.21
[15:44:26][MKT] E(Rm) = 11.2880%  ERP = 7.0000%
[15:44:26][MKT]   → Damodaran 한국 ERP 7.0% 적용
[15:44:26][MKT]   (참고) KOSPI 10년 기하평균 = 12.8238%

  Rf  =  4.288%    (BOK 10Y 국고채)
  ERP =  7.000%    (방식: damodaran_floor)
  E(Rm) = 11.288%


## Cell 6 · `load_korea_revenue_forecast` 패치

구스키마 created_at 버그(매출 forecast 상수 복제) 수정판. updated_at 버전 선택 + 3중 검증(분기 수·연속성·상수 시계열).

> ⚠️ 만약 Cell 3(import 셀)을 다시 실행하면 패치가 풀리므로, 이 셀도 반드시 재실행하세요.

In [6]:
# ═══════════════════════════════════════════════════════════════
#  PATCH 셀 · load_korea_revenue_forecast 교체
#  (구스키마 created_at 버그 → 매출 forecast 상수 복제 문제 수정)
#
#  ▸ 붙여넣을 위치 : korea_fcff_dcf_valuation_v7.ipynb 의
#                    import 셀(DATA.korea_valuation_helpers import) "바로 다음".
#                    KoreaDCFModel 실행(Cell 9/10) 전이면 어디든 OK.
#  ▸ 원리          : 노트북 전역의 load_korea_revenue_forecast 이름을
#                    아래 수정판으로 재바인딩 → KoreaDCFModel.load_sales 가
#                    호출 시점에 이 패치판을 사용.
#  ▸ 주의          : import 셀을 다시 실행하면 패치가 풀리므로,
#                    import 셀 재실행 후에는 이 셀도 재실행할 것.
#
#  ▸ 수정 내용 (korea_revenue_analysis_notebook_v3 의 fetch_forecast 와 동일 원리)
#    1) 버전 식별: MAX(created_at) → MAX(updated_at) (자동 감지, 없으면 created_at)
#       - 구스키마 UNIQUE KEY(date,ticker,indicator) 에서는 재예측 시
#         겹치는 분기의 created_at 이 갱신되지 않아 '새로 생긴 분기 1개'만
#         최신 버전으로 잡히는 문제가 있었음 (예: A031330 → 2028Q1 한 분기).
#    2) 같은 분기 중복 시 가장 최근 갱신분만 사용.
#    3) 검증 추가 — 아래 셋 중 하나라도 걸리면 ValueError 로 즉시 차단
#       (잘못된 valuation 이 DB 에 저장되는 것을 막기 위함):
#       a. 분기 수 < horizon
#       b. 분기 불연속 (중간 분기 누락)
#       c. 전 분기 동일값 (상수 시계열 — 이번 버그의 증상)
# ═══════════════════════════════════════════════════════════════
from sqlalchemy import text as _sa_text

_FC_MODEL_PRIORITY = ("Ensemble", "SARIMA", "ETS", "Theta")
_FC_RECENCY_CACHE: dict = {}


def _fc_recency_col(table_name: str) -> str:
    """updated_at 컬럼이 있으면 그것을, 없으면 created_at 을 버전 기준으로 사용."""
    if table_name in _FC_RECENCY_CACHE:
        return _FC_RECENCY_CACHE[table_name]
    sql = ("SELECT COLUMN_NAME FROM information_schema.COLUMNS "
           "WHERE TABLE_SCHEMA = :db AND TABLE_NAME = :tbl")
    with engine.connect() as conn:
        cols = {r[0].lower() for r in conn.execute(
            _sa_text(sql), {"db": db_info["database"], "tbl": table_name}
        ).fetchall()}
    rc = "updated_at" if "updated_at" in cols else "created_at"
    _FC_RECENCY_CACHE[table_name] = rc
    if rc == "created_at":
        log("PATCH", f"[주의] {table_name} 에 updated_at 없음 → created_at 기준. "
                     "구스키마면 일부 종목 분기 누락 가능 — updated_at 추가 권장.")
    return rc


def load_korea_revenue_forecast(ticker: str,
                                db_info: dict,
                                table_name: str = "korea_revenue_forecast_result",
                                horizon: int = 8):
    """[PATCHED v7-fix] 최신 '실행 버전' 전체 분기를 안전하게 로드.

    Returns
    -------
    (pd.Series, str, str)
        forecast : 천원 단위, index=분기 date (호출측에서 ×1000 변환)
        model    : 사용된 indicator (Ensemble 우선)
        run_date : 예측 실행 버전 날짜 (YYYY-MM-DD)
    """
    rc = _fc_recency_col(table_name)

    df, used_model, run_date = None, "", None
    for model in _FC_MODEL_PRIORITY:
        # indicator 가 'Ensemble' 형태든 '매출액(천원)_Ensemble' 형태든 모두 매칭
        with engine.connect() as conn:
            row = conn.execute(_sa_text(
                f"SELECT MAX(DATE({rc})) FROM {table_name} "
                f"WHERE ticker = :t AND (indicator = :m OR indicator LIKE :ml)"
            ), {"t": ticker, "m": model, "ml": f"%\\_{model}"}).fetchone()
        if row is None or row[0] is None:
            continue
        run_date = str(row[0])
        cand = pd.read_sql(_sa_text(
            f"SELECT date, value, {rc} AS run_ts FROM {table_name} "
            f"WHERE ticker = :t AND (indicator = :m OR indicator LIKE :ml) "
            f"  AND DATE({rc}) = :fd "
            f"ORDER BY date"
        ), engine, params={"t": ticker, "m": model,
                           "ml": f"%\\_{model}", "fd": run_date})
        if not cand.empty:
            df, used_model = cand, model
            break

    if df is None:
        return pd.Series(dtype=float), "", None

    df["date"]   = pd.to_datetime(df["date"])
    df["run_ts"] = pd.to_datetime(df["run_ts"])
    df = (df.sort_values(["date", "run_ts"])
            .drop_duplicates(subset=["date"], keep="last"))
    fc = df.set_index("date")["value"].astype(float).sort_index()

    # ── 검증 a: 분기 수 ───────────────────────────────────────
    if len(fc) < horizon:
        raise ValueError(
            f"[{ticker}] forecast 분기 {len(fc)}개 < horizon {horizon} "
            f"(version={rc} {run_date}, indicator={used_model}). "
            f"구스키마 created_at 잔존 가능성 — 해당 종목 재예측 또는 "
            f"테이블 updated_at 마이그레이션 필요."
        )
    fc = fc.iloc[:horizon]

    # ── 검증 b: 분기 연속성 ──────────────────────────────────
    per = fc.index.to_period("Q")
    gaps = (per[1:].astype("int64") - per[:-1].astype("int64"))
    if (gaps != 1).any():
        bad = [str(per[i + 1]) for i, g in enumerate(gaps) if g != 1]
        raise ValueError(
            f"[{ticker}] forecast 분기 불연속: {bad} "
            f"(version={rc} {run_date}). 해당 종목 재예측 필요."
        )

    # ── 검증 c: 상수 시계열 (이번 버그의 증상) ────────────────
    if fc.nunique() == 1:
        raise ValueError(
            f"[{ticker}] forecast {horizon}개 분기가 모두 동일값"
            f"({fc.iloc[0]:,.0f}천원) — 단일 분기 복제 의심. "
            f"(version={rc} {run_date}). 해당 종목 재예측 필요."
        )

    return fc, used_model, run_date


# log("PATCH", "load_korea_revenue_forecast → updated_at 버전 선택 + 3중 검증 패치 적용 완료")

## Cell 7 · KoreaDCFModel v8 클래스 & `process_one_ticker_kr`

v8 원본과 동일 — Rd≥Rf · WACC≥max(5%,Rf+1%) · g_term≤min(GDP,Rf) · TV 스프레드 2%+배수 35배 · Sanity Guard(±30배) · g0 기반 ρ 적응형 상한

In [7]:
# ═══════════════════════════════════════════════════════════════
#  KoreaDCFModel v8  —  한국 주식 Sales-driven FCFF DCF
#  (US v11 v8 + v9.2 패치 + ★ US v13 Validity 가드 이식)
#  ─────────────────────────────────────────────────────────────────
#  v7 → v8 변경 (US v13 Validity Patch 이식):
#    ② compute_cost_of_debt : Rd ≥ Rf 하한
#       compute_wacc        : WACC 하한 = max(5%, Rf+1%)  (구 절대 5%)
#    ③ compute_terminal_growth : g_term ≤ min(GDP, Rf)
#       compute_valuation      : 최소 스프레드 2%(구 1%) + TV/FCFF 배수 ≤ 35배
#    ④ compute_valuation       : Sanity Guard (지분가치/시총 ±30배)
#    ⑤ _estimate_phase2_growth : g0 기반 ρ 적응형 상한
# ═══════════════════════════════════════════════════════════════

class KoreaDCFModel:
    """
    한국 기업 FCFF DCF Valuation Model — v8 (US v13 Validity 가드 이식).

    사용 흐름:
        m = KoreaDCFModel(ticker="A005930", engine=engine, db_info=db_info,
                          rf=RF, e_rm=E_RM, kospi_series=KOSPI_PX, verbose=True)
        m.run()
        m.save_to_db("2026-04-21")
        m.plot()

    v1 → v2 변경사항:
      [v8 패치]
        - NWC_FIELD_ALIASES 매핑 + _compute_bs_nwc_series()
        - estimate_nwc_coef() 가 _last_actual_nwc 저장
        - compute_fcff() prev_nwc 를 γ × last_actual_sales 로 시드
        - _compute_historical_fcff() 첫 분기 prev_nwc 를 γ × first_sales 로 시드
      [v9.2 패치]
        - _estimate_phase2_growth(): 다중 fallback 체인
          · Method 1: 8Q-CAGR (양수 케이스만)
          · Method 2: 12Q-TTM rolling (★ 신규, 부호 반전 명시 처리)
          · Method 3: Forecast yr1/yr2 (★ 신규, g0_fc 로 항상 계산)
        - g0_hist + g0_fc 결합 로직 (min, forecast 우선, 부호 불일치)
        - 음수 g₀ 허용 (G_FLOOR_MIN = -0.20)
        - ρ OLS 보정 (음수 annual 도 회귀 포함)
        - AR(1) clip 하한 G_FLOOR_MIN 으로 완화

    한국 특화 (v1 동일):
      - FMP API 호출 없음 → 모두 DB 조회
      - 자체 베타 계산 (KSE_Price + KOSPI 일별수익률 10년 회귀, Blume)
      - NWC ≈ Receivables + Inventories - Short-term debts  (한국 BS 보완)
      - DataQualityReport 로 모든 fallback 추적
    """

    # ─────────────────────────────────────────────────────────────
    # ★ v8 패치: NWC 필드 alias 매핑 (silent 0 대체 방지)
    #
    # 한국 DataGuide 의 정규화된 컬럼명을 표준 키 → 후보 컬럼들로 매핑.
    # 핵심 컬럼이 없거나 전부 0 이면 verbose 모드에서 경고 출력.
    # ─────────────────────────────────────────────────────────────
    NWC_FIELD_ALIASES = {
        # 자산 측
        "receivables":      ["receivables", "trade_receivables",
                             "accounts_receivable"],
        "inventories":      ["inventories", "inventory"],
        # 부채 측 (단기성)
        "short_term_debt":  ["short_term_debt", "short_term_borrowings"],
        "current_lt_debt":  ["current_lt_debt", "current_portion_of_lt_debt"],
        "lease_liab":       ["lease_liab", "lease_liabilities", "lease_liability"],
    }

    # ─────────────────────────────────────────────────────────────
    # ★ v7: 영업운전자본 alias (재무부채 제외 — 순수 영업 항목만)
    #   자산 측: 매출채권 + 재고자산 + 선급비용
    #   부채 측: 매입채무 + 미지급비용 + 미지급금 + 선수금 + 계약부채
    #   각 std_key 의 첫 후보가 helper(DG_ITEM_CODES) 권장 키와 일치.
    # ─────────────────────────────────────────────────────────────
    NWC_OP_ASSET_ALIASES = {
        "receivables":       ["receivables", "trade_receivables", "accounts_receivable"],
        "inventories":       ["inventories", "inventory"],
        "prepaid_expenses":  ["prepaid_expenses", "prepaid", "prepaid_exp"],
    }
    NWC_OP_LIAB_ALIASES = {
        "payables":             ["payables", "trade_payables", "accounts_payable"],
        "accrued_expenses":     ["accrued_expenses", "accrued_exp", "accrued"],
        "other_payables":       ["other_payables", "non_trade_payables", "other_payable"],
        "advances_received":    ["advances_received", "advance_received", "customer_advances", "unearned"],
        "contract_liabilities": ["contract_liabilities", "contract_liab", "deferred_revenue"],
    }

    def __init__(self, ticker, engine, db_info,
                 rf, e_rm, kospi_series,
                 forecast_horizon=FORECAST_HORIZON,
                 min_history=MIN_HISTORY,
                 gdp_growth=GDP_GROWTH,
                 verbose=False):
        self.ticker_dg     = to_dg_ticker(ticker)
        self.ticker_price  = to_price_ticker(ticker)
        self.engine        = engine
        self.db_info       = db_info
        self.rf            = rf
        self.e_rm          = e_rm
        self.erp           = e_rm - rf
        self.kospi         = kospi_series
        self.horizon       = forecast_horizon
        self.min_history   = min_history
        self.gdp_growth    = gdp_growth
        self.verbose       = verbose

        # 데이터 저장소
        self._sales_actual:   Optional[pd.Series]    = None
        self._sales_forecast: Optional[pd.Series]    = None
        self._used_model:     str                    = ""
        self._forecast_date:  Optional[date]         = None
        self._fs_wide:        Optional[pd.DataFrame] = None
        self._stock_data:     Optional[pd.DataFrame] = None
        self._fcff_history:   Optional[pd.Series]    = None
        self._beta_info:      Optional[Dict]         = None
        self._mkt_cap:        Optional[float]        = None
        self._current_price:  Optional[float]        = None
        self._last_actual_nwc: Optional[float]       = None  # ★ v8
        self._nwc_method:      str                    = "?"      # ★ v7
        self._op_ca_latest:    float                  = np.nan   # ★ v7
        self._op_cl_latest:    float                  = np.nan   # ★ v7
        self._nwc_to_sales:    float                  = np.nan   # ★ v7

        self.result_df: Optional[pd.DataFrame]       = None
        self.valuation: Optional[Dict]               = None
        self.report = DataQualityReport(ticker=self.ticker_dg)

    # ─────────────────────────────────────────────────────────
    # 1. 데이터 로드
    # ─────────────────────────────────────────────────────────

    def load_sales(self):
        """매출 actual + forecast 로드. 핵심항목이라 결측 시 즉시 raise."""
        # Actual
        wide = load_korea_financials_wide(
            self.ticker_dg, self.db_info, table_name=TABLE_FS,
            item_keys=["revenue"], fillna_zero=False,
        )
        actual = wide["revenue"].dropna() * FS_UNIT_MULTIPLIER  # 천원 → 원
        if actual.empty:
            self.report.add("revenue_actual", "missing", n_obs=0,
                            note="korea_fs_data_from_DG에 매출 데이터 없음")
            raise ValueError(f"[{self.ticker_dg}] Sales actual 데이터 없음")

        if len(actual) < MIN_REVENUE_QUARTERS:
            self.report.add("revenue_actual", "missing", n_obs=len(actual),
                            note=f"매출 {len(actual)}분기 < 최소 {MIN_REVENUE_QUARTERS}")
            raise ValueError(
                f"[{self.ticker_dg}] 매출 {len(actual)}분기 < 최소 {MIN_REVENUE_QUARTERS}분기"
            )
        self.report.add("revenue_actual", "ok", n_obs=len(actual))

        # Forecast
        forecast, model_name, ca = load_korea_revenue_forecast(
            self.ticker_dg, self.db_info, table_name=TABLE_FORECAST,
            horizon=self.horizon,
        )
        if forecast.empty:
            self.report.add("revenue_forecast", "missing", n_obs=0,
                            note="korea_revenue_forecast_result에 예측 없음")
            raise ValueError(
                f"[{self.ticker_dg}] 매출 forecast 없음 — Korea_revenue_forecast 먼저 실행"
            )
        forecast = forecast * FS_UNIT_MULTIPLIER  # 천원 → 원
        self.report.add("revenue_forecast", "ok", n_obs=len(forecast),
                        note=f"model={model_name}, created_at={ca}")

        self._sales_actual   = actual
        self._sales_forecast = forecast.iloc[:self.horizon]
        self._used_model     = model_name
        self._forecast_date  = ca

        if self.verbose:
            log(self.ticker_dg,
                f"Sales actual={len(actual)}Q forecast={len(self._sales_forecast)}Q "
                f"model={model_name} created_at={ca}")
        return self

    def load_financials(self):
        """IS / BS / CF wide-form + 주식수 로드 + (★ v8) NWC 핵심 필드 sanity check."""
        # 재무 항목 wide-form
        wide = load_korea_financials_wide(
            self.ticker_dg, self.db_info, table_name=TABLE_FS,
            item_keys=None,  # 전체
            fillna_zero=False,
        )
        if wide.empty:
            raise ValueError(f"[{self.ticker_dg}] 재무 데이터 wide-form 비어있음")

        # 단위 변환: 천원 → 원 (주식수만 제외)
        non_share_cols = [c for c in wide.columns
                          if c not in ("shares_treasury_adj", "shares_common")]
        wide[non_share_cols] = wide[non_share_cols] * FS_UNIT_MULTIPLIER

        # 핵심항목 검증 (영업이익은 필수)
        if "operating_income" not in wide.columns or wide["operating_income"].dropna().empty:
            self.report.add("operating_income", "missing", n_obs=0,
                            note="영업이익 데이터 없음 — 가치평가 불가")
            raise ValueError(f"[{self.ticker_dg}] operating_income 데이터 없음")
        self.report.add("operating_income", "ok",
                        n_obs=int(wide["operating_income"].notna().sum()))

        # ★ v8: NWC 핵심 필드 sanity check
        for std_name, aliases in self.NWC_FIELD_ALIASES.items():
            present = [a for a in aliases if a in wide.columns]
            if not present:
                if self.verbose:
                    log(self.ticker_dg,
                        f"⚠️  NWC 필드 없음: {std_name} (aliases={aliases})")
            else:
                col = pd.to_numeric(wide[present[0]], errors="coerce")
                if col.abs().sum() == 0:
                    if self.verbose:
                        log(self.ticker_dg,
                            f"⚠️  NWC 필드가 전부 0: {present[0]} ({std_name})")

        self._fs_wide = wide
        if self.verbose:
            log(self.ticker_dg, f"FS wide  shape={wide.shape}  "
                f"기간={wide.index.min().date()} ~ {wide.index.max().date()}")
        return self

    # ─────────────────────────────────────────────────────────
    # 2. 변수별 추정
    # ─────────────────────────────────────────────────────────

    @staticmethod
    def _winsorize(s, limits=WINSORIZE_LIMITS):
        s = s.dropna()
        if len(s) < 4:
            return s
        lo, hi = s.quantile(limits[0]), s.quantile(limits[1])
        return s.clip(lo, hi)

    @staticmethod
    def _ols_ratio(x, y):
        mask = x.notna() & y.notna() & (x != 0)
        if mask.sum() < OLS_MIN_SAMPLES:
            return np.nan, -1.0, int(mask.sum())
        slope, _, r, _, _ = stats.linregress(x[mask], y[mask])
        return float(slope), float(r ** 2), int(mask.sum())

    def _resolve_alias(self, std_name: str) -> Optional[str]:
        """★ v8: NWC_FIELD_ALIASES 의 std_name 에 대해 wide 컬럼에 존재하는 첫 alias 반환."""
        wide = self._fs_wide
        if wide is None:
            return None
        for alias in self.NWC_FIELD_ALIASES.get(std_name, [std_name]):
            if alias in wide.columns:
                return alias
        return None

    def _resolve_from(self, alias_map: dict, std_name: str) -> Optional[str]:
        """★ v7: 임의 alias_map 에서 wide 컬럼에 존재하는 첫 alias 반환."""
        wide = self._fs_wide
        if wide is None:
            return None
        for alias in alias_map.get(std_name, [std_name]):
            if alias in wide.columns:
                return alias
        return None

    def _sum_components(self, alias_map: dict):
        """★ v7: alias_map 의 각 std_key 를 resolve → 결측 0 으로 합산.
        반환: (합산 Series, 실제 resolve 된 std_key 리스트)."""
        wide = self._fs_wide
        total = pd.Series(0.0, index=wide.index)
        resolved = []
        for std_name in alias_map:
            col = self._resolve_from(alias_map, std_name)
            if col is not None:
                total = total + pd.to_numeric(wide[col], errors="coerce").fillna(0)
                resolved.append(std_name)
        return total, resolved

    def estimate_tax_rate(self):
        """실효세율 (pretaxIncome > 0 분기, clip 0~40%, 한국 법정세율 22% fallback)"""
        df = self._fs_wide[["pretax_income", "tax_expense"]].dropna()
        df = df[df["pretax_income"] > 0]
        if df.empty:
            self.report.add("tax_rate", "fallback_zero", n_obs=0,
                            value=0.22, note="법정세율 22% fallback")
            return 0.22
        rates = (df["tax_expense"] / df["pretax_income"]).clip(0, 0.40)
        med = float(rates.median())
        self.report.add("tax_rate", "ok", n_obs=len(rates), value=med)
        return med

    def estimate_opm(self, sales_series):
        """OPM 예측: SARIMA/ETS/Theta Ensemble."""
        df = self._fs_wide[["operating_income", "revenue"]].dropna()
        df = df[df["revenue"] > 0]
        df["opm"] = df["operating_income"] / df["revenue"]
        df["opm"] = self._winsorize(df["opm"])
        opm_series = df["opm"].dropna()

        if len(opm_series) < self.min_history:
            fallback = float(opm_series.median()) if not opm_series.empty else 0.05
            self.report.add("opm_forecast", "fallback_median",
                            n_obs=len(opm_series), value=fallback,
                            note=f"OPM history {len(opm_series)} < {self.min_history}")
            return pd.Series([fallback] * len(sales_series), index=sales_series.index)

        try:
            freq_alias = infer_freq_alias(opm_series.index)
            m = seasonal_periods_from_freq(freq_alias)
        except Exception:
            m = 4

        forecasts = {}
        for name, fn in [
            ("SARIMA", lambda y: forecast_sarima(y, self.horizon, seasonal_period=m)),
            ("ETS",    lambda y: forecast_ets(y, self.horizon, m=m)),
            ("Theta",  lambda y: forecast_theta(y, self.horizon, m=m)),
        ]:
            try:
                res = fn(opm_series)
                if "forecast" in res and "error" not in res:
                    forecasts[name] = np.asarray(res["forecast"])
            except Exception:
                pass

        if forecasts:
            ens = np.mean(list(forecasts.values()), axis=0)
            ens = np.clip(ens, -0.30, 0.50)  # OPM 범위 안정화
            self.report.add("opm_forecast", "ok",
                            n_obs=len(opm_series),
                            note=f"models={list(forecasts.keys())}")
            return pd.Series(ens, index=sales_series.index)

        # Fallback
        fallback = float(opm_series.tail(8).median()
                         if len(opm_series) >= 8 else opm_series.median())
        fallback = float(np.clip(fallback, -0.30, 0.50))
        self.report.add("opm_forecast", "fallback_median",
                        n_obs=len(opm_series), value=fallback,
                        note="모든 ts 모델 실패")
        return pd.Series([fallback] * len(sales_series), index=sales_series.index)

    def estimate_ratio_coef(self, target_keys, ratio_name, take_abs=False):
        """
        D&A / CapEx 비율 계수 추정.
        target_keys : 합산할 컬럼 리스트 (예: D&A = ['da_cf', 'intangible_amort_cf'])
        OLS slope 우선 → R²<0.30 or 부족 → median ratio fallback
        """
        # Sales
        sales = self._fs_wide["revenue"].dropna()
        sales = sales[sales > 0]
        if sales.empty:
            self.report.add(ratio_name, "fallback_zero", n_obs=0, value=0.0,
                            note="revenue 없음")
            return 0.0, "missing"

        # Target value (여러 컬럼 합산, 결측은 0으로)
        target = pd.Series(0.0, index=self._fs_wide.index)
        valid_cols = []
        for k in target_keys:
            if k in self._fs_wide.columns:
                col = self._fs_wide[k].fillna(0.0)
                target = target + col
                valid_cols.append(k)
        if not valid_cols:
            self.report.add(ratio_name, "fallback_zero", n_obs=0, value=0.0,
                            note=f"{target_keys} 모두 컬럼 없음")
            return 0.0, "missing"
        if take_abs:
            target = target.abs()

        merged = pd.concat([sales.rename("rev"), target.rename("y")],
                           axis=1, join="inner").dropna()
        merged = merged[merged["rev"] > 0]
        if merged.empty:
            self.report.add(ratio_name, "fallback_zero", n_obs=0, value=0.0,
                            note="merged 비어있음")
            return 0.0, "empty"

        slope, r2, n = self._ols_ratio(merged["rev"], merged["y"])
        use_ols = (not np.isnan(slope) and r2 >= OLS_MIN_R2 and slope >= 0
                   and n >= OLS_MIN_SAMPLES)

        if use_ols:
            self.report.add(ratio_name, "ok", n_obs=n, value=slope, r2=r2,
                            note=f"OLS cols={valid_cols}")
            return float(slope), "ols"
        # median fallback
        ratios = (merged["y"] / merged["rev"]).replace([np.inf, -np.inf], np.nan).dropna()
        if ratios.empty:
            self.report.add(ratio_name, "fallback_zero", n_obs=0, value=0.0)
            return 0.0, "ratio_empty"
        ratios = self._winsorize(ratios)
        med = float(ratios.median())
        self.report.add(ratio_name, "fallback_median", n_obs=len(ratios),
                        value=med, r2=r2,
                        note=f"OLS R²={r2:.2f} → median, cols={valid_cols}")
        return max(med, 0.0), "median_ratio"

    # ─────────────────────────────────────────────────────────
    # ★ v8 패치: NWC 시계열 계산 (alias 매핑 + 표준 산식)
    # ─────────────────────────────────────────────────────────
    def _compute_operating_nwc_series(self):
        """
        ★ v7: 비현금 영업운전자본 시계열.
            NWC = (매출채권 + 재고자산 + 선급비용)
                  − (매입채무 + 미지급비용 + 미지급금 + 선수금 + 계약부채)
        재무부채(단기차입금/유동성장기부채/리스)·현금은 제외 — 순수 영업 운전자본.
        반환: (DataFrame[date, nwc, op_ca, op_cl], 자산 resolve, 부채 resolve, 부채존재여부)
        """
        wide = self._fs_wide
        op_ca, ca_res = self._sum_components(self.NWC_OP_ASSET_ALIASES)
        op_cl, cl_res = self._sum_components(self.NWC_OP_LIAB_ALIASES)
        has_op_liab = len(cl_res) > 0     # 신규 영업부채 항목이 하나라도 있으면 v7 적용
        nwc = op_ca - op_cl
        self._op_ca_latest = float(op_ca.iloc[-1]) if len(op_ca) else np.nan
        self._op_cl_latest = float(op_cl.iloc[-1]) if len(op_cl) else np.nan
        df = pd.DataFrame({"date": wide.index, "nwc": nwc.values}).reset_index(drop=True)
        return df, ca_res, cl_res, has_op_liab

    def _compute_legacy_nwc_series(self) -> pd.DataFrame:
        """
        (구 v6 proxy) NWC = (Recv+Inv) − (단기차입금+유동성장기부채+리스).
        ※ 재무부채 혼입 + 영업매입채무 누락 → 이론적으로 부정확. 폴백/비교용으로만 유지.
        """
        wide = self._fs_wide
        recv_col = self._resolve_alias("receivables")
        inv_col  = self._resolve_alias("inventories")
        rec = pd.to_numeric(wide[recv_col], errors="coerce").fillna(0) if recv_col else pd.Series(0.0, index=wide.index)
        inv = pd.to_numeric(wide[inv_col],  errors="coerce").fillna(0) if inv_col  else pd.Series(0.0, index=wide.index)
        st = pd.Series(0.0, index=wide.index)
        for std_name in ["short_term_debt", "current_lt_debt", "lease_liab"]:
            col = self._resolve_alias(std_name)
            if col:
                st = st + pd.to_numeric(wide[col], errors="coerce").fillna(0)
        nwc = (rec + inv) - st
        self._op_ca_latest = float((rec + inv).iloc[-1]) if len(wide) else np.nan
        self._op_cl_latest = float(st.iloc[-1]) if len(wide) else np.nan
        return pd.DataFrame({"date": wide.index, "nwc": nwc.values}).reset_index(drop=True)

    def _compute_bs_nwc_series(self) -> pd.DataFrame:
        """
        NWC 시계열 디스패처 (★ v7).
          USE_OPERATING_NWC=True 이고 신규 영업부채 항목이 DB 에 적재돼 있으면
          → 영업운전자본(operating_v7). 미적재면 → legacy proxy 로 자동 폴백.
        """
        if USE_OPERATING_NWC:
            df, ca_res, cl_res, has_op_liab = self._compute_operating_nwc_series()
            if has_op_liab:
                self._nwc_method = "operating_v7"
                self.report.add("nwc_definition", "ok", n_obs=len(df),
                                note=f"operating_v7 CA={ca_res} CL={cl_res}")
                if self.verbose:
                    log(self.ticker_dg,
                        f"NWC[v7 영업] CA={ca_res} CL={cl_res}  "
                        f"최근 CA={self._op_ca_latest/1e9:,.1f}B CL={self._op_cl_latest/1e9:,.1f}B "
                        f"NWC={(self._op_ca_latest-self._op_cl_latest)/1e9:,.1f}B")
                return df
            self.report.warn("v7 영업부채 항목 없음(DB 미적재?) → legacy NWC proxy 폴백")
            if self.verbose:
                log(self.ticker_dg, "⚠️  v7 영업부채 항목 없음 → legacy proxy 사용")
        self._nwc_method = "legacy_proxy"
        return self._compute_legacy_nwc_series()

    def estimate_nwc_coef(self):
        """
        한국 BS는 유동부채 컬럼 없음 → 보완:
          NWC ≈ (Receivables + Inventories) - (Short-term debt + Current LT debt + Lease)
        NWC/Sales 비율 OLS 또는 median.

        ★ v8: 마지막 실측 NWC 를 self._last_actual_nwc 에 저장 → ΔNWC 시드값
              (γ × last_actual_sales) 와의 정합성 진단용.
        """
        nwc_df = self._compute_bs_nwc_series()

        # ★ v8: 마지막 실측 NWC 저장
        self._last_actual_nwc = (float(nwc_df["nwc"].iloc[-1])
                                 if not nwc_df.empty else 0.0)

        sales = self._fs_wide["revenue"]
        sales_df = pd.DataFrame({"date": sales.index, "rev": sales.values}).reset_index(drop=True)
        merged = sales_df.merge(nwc_df, on="date", how="inner").dropna()
        merged = merged[merged["rev"] > 0]
        if merged.empty:
            self.report.add("nwc", "fallback_zero", n_obs=0, value=0.0,
                            note="merged 비어있음")
            return 0.0, "empty"

        slope, r2, n = self._ols_ratio(merged["rev"], merged["nwc"])
        use_ols = (not np.isnan(slope) and r2 >= OLS_MIN_R2
                   and n >= OLS_MIN_SAMPLES)
        if use_ols:
            self.report.add("nwc", "ok", n_obs=n, value=slope, r2=r2,
                            note="(Recv+Inv) - (ST debts) ratio")
            return float(slope), "ols"
        ratios = (merged["nwc"] / merged["rev"]).replace(
            [np.inf, -np.inf], np.nan).dropna()
        if ratios.empty:
            self.report.add("nwc", "fallback_zero", n_obs=0, value=0.0)
            return 0.0, "ratio_empty"
        med = float(self._winsorize(ratios).median())
        self.report.add("nwc", "fallback_median", n_obs=len(ratios),
                        value=med, r2=r2,
                        note=f"OLS R²={r2:.2f} → median ratio")
        return med, "median_ratio"

    # ─────────────────────────────────────────────────────────
    # 3. FCFF 계산
    # ─────────────────────────────────────────────────────────

    def compute_fcff(self):
        """Sales-driven FCFF = NOPAT + D&A - CapEx - ΔNWC.

        ★ v8 패치 (vs v1):
          v1: prev_nwc=None → 첫 예측분기 ΔNWC=0 → FCFF 과대계상
          v2: prev_nwc = γ × last_actual_sales → 첫 분기 ΔNWC 도 정합
        """
        sales_fc = self._sales_forecast
        tax      = self.estimate_tax_rate()
        opm_fc   = self.estimate_opm(sales_fc)

        alpha, _ = self.estimate_ratio_coef(
            ["da_cf", "intangible_amort_cf"], "da")
        beta_, _ = self.estimate_ratio_coef(
            ["capex_tangible", "capex_intangible"], "capex", take_abs=True)
        gamma, _ = self.estimate_nwc_coef()
        self._nwc_to_sales = gamma   # ★ v7: 결과 저장용

        # ★ v8: ΔNWC 시드값 = γ × 마지막 실제 매출 (v1: None)
        last_actual_sales = (float(self._sales_actual.iloc[-1])
                             if (self._sales_actual is not None
                                 and not self._sales_actual.empty)
                             else 0.0)
        prev_nwc = gamma * last_actual_sales

        # ★ v8: 실측 NWC 와 γ×sales 모델의 정합성 진단 로그
        if self.verbose and abs(last_actual_sales) > 0:
            actual_nwc = self._last_actual_nwc
            if actual_nwc is not None and abs(prev_nwc) > 0:
                ratio_gap = abs(actual_nwc - prev_nwc) / max(abs(actual_nwc), 1)
                if ratio_gap > 0.5:
                    log(self.ticker_dg,
                        f"[NWC seed] γ×sales={prev_nwc/1e9:,.1f}B vs "
                        f"실측={actual_nwc/1e9:,.1f}B 괴리 {ratio_gap:.1%}")

        rows = []
        for i, (dt, sales) in enumerate(sales_fc.items()):
            opm   = float(opm_fc.iloc[i])
            ebit  = sales * opm
            nopat = ebit * (1 - tax)
            da    = alpha * sales
            capex = beta_ * sales
            nwc   = gamma * sales
            delta_nwc = nwc - prev_nwc
            prev_nwc = nwc

            fcff = nopat + da - capex - delta_nwc
            invested = capex + max(delta_nwc, 0)
            roic = nopat / invested if invested > 1e-6 else np.nan
            reinv = capex / nopat if nopat > 1e-6 else np.nan

            rows.append({
                "date": dt,
                "quarter": f"{dt.year}Q{dt.quarter}",
                "sales_forecast": sales, "opm_forecast": opm,
                "ebit": ebit, "tax_rate": tax, "nopat": nopat,
                "da": da, "capex": capex, "nwc": nwc,
                "delta_nwc": delta_nwc, "fcff": fcff, "roic": roic,
                "reinvestment_rate": reinv,
            })
        self.result_df = pd.DataFrame(rows)
        self._compute_historical_fcff(tax, alpha, beta_, gamma)
        return self

    def _compute_historical_fcff(self, tax, alpha, beta_, gamma):
        """과거 FCFF 시계열 (terminal CAGR / 시각화용).

        ★ v8 패치: 첫 분기 ΔNWC = γ × first_sales 시드값
                   (v1: None → 첫 분기 ΔNWC=0 → CAGR 왜곡)
        """
        sales_act = self._sales_actual
        if sales_act is None or sales_act.empty:
            self._fcff_history = pd.Series(dtype=float); return

        wide = self._fs_wide
        rows = []
        # ★ v8: 첫 분기 ΔNWC 시드값 = γ × 첫 sales
        first_sales = float(sales_act.iloc[0]) if not sales_act.empty else 0.0
        prev_nwc = gamma * first_sales

        for dt, sales in sales_act.items():
            if dt not in wide.index:
                continue
            opm_row = wide.loc[dt].get("operating_income", np.nan)
            rev_row = wide.loc[dt].get("revenue", np.nan)
            if pd.isna(opm_row) or pd.isna(rev_row) or rev_row <= 0:
                prev_nwc = gamma * sales; continue
            opm   = float(opm_row) / float(rev_row)
            ebit  = sales * opm
            nopat = ebit * (1 - tax)
            da    = alpha * sales
            capex = beta_ * sales
            nwc   = gamma * sales
            dnwc  = nwc - prev_nwc
            prev_nwc = nwc
            rows.append({"date": dt, "fcff": nopat + da - capex - dnwc})
        if rows:
            self._fcff_history = pd.DataFrame(rows).set_index("date")["fcff"]
        else:
            self._fcff_history = pd.Series(dtype=float)

    # ─────────────────────────────────────────────────────────
    # 4. WACC 계산 (한국 시장 어댑테이션) — v1 동일
    # ─────────────────────────────────────────────────────────

    def compute_beta_re(self):
        """
        베타 + Re (CAPM):
          β_blume = 0.67×|β| + 0.33  (호영님 요청)
          Re      = Rf + β_blume × ERP
        """
        info = compute_beta_10y(
            self.ticker_dg, self.db_info, kospi_series=self.kospi,
            years=10, min_obs=750,
        )
        self._beta_info = info

        if np.isnan(info["beta_raw"]):
            beta_blume = 1.0  # fallback
            self.report.add("beta", "fallback_median", n_obs=info["n_obs"],
                            value=beta_blume,
                            note="베타 계산 실패 → β=1.0 fallback")
        else:
            beta_blume = info["beta_blume"]
            self.report.add("beta", "ok", n_obs=info["n_obs"],
                            value=beta_blume, r2=info["r_squared"],
                            note=f"β_raw={info['beta_raw']:.3f} → blume={beta_blume:.3f}")

        re = self.rf + beta_blume * self.erp
        if self.verbose:
            log(self.ticker_dg,
                f"β_raw={info.get('beta_raw', np.nan):.3f} "
                f"β_blume={beta_blume:.3f}  Re={re:.4%}  "
                f"(n={info['n_obs']}, R²={info.get('r_squared', np.nan):.2f})")
        return float(re), beta_blume

    def compute_cost_of_debt(self, tax):
        """이자비용 / 평균 총부채. 실패 시 RD_DEFAULT."""
        wide = self._fs_wide
        if "interest_expense" not in wide.columns:
            self.report.add("rd", "fallback_zero", value=RD_DEFAULT,
                            note="interest_expense 컬럼 없음")
            return RD_DEFAULT
        # 총부채 = 단기차입금 + 유동성장기부채 + 사채 + 장기차입금 + 리스
        total_debt = pd.Series(0.0, index=wide.index)
        for k in DEBT_KEYS:
            if k in wide.columns:
                total_debt = total_debt + wide[k].fillna(0)
        td_avg = (total_debt + total_debt.shift(1)) / 2
        ie = wide["interest_expense"].abs()

        valid = (td_avg > 0) & ie.notna()
        if valid.sum() < 4:
            self.report.add("rd", "fallback_zero", n_obs=int(valid.sum()),
                            value=RD_DEFAULT, note="유효 분기 < 4")
            return RD_DEFAULT
        rd_series = (ie[valid] / td_avg[valid]).clip(0, 0.20)
        rd = float(rd_series.median())
        # ★ v8: Rd 하한 = Rf — 장부 이자율(변동금리·구채권 효과)이 무위험수익률보다
        #        낮으면 forward-looking 부채비용으로 부적절 (채권자도 최소 Rf 요구)
        if V8_RD_FLOOR_RF and rd < self.rf:
            self.report.add("rd", "ok", n_obs=int(valid.sum()), value=float(self.rf),
                            note=f"median {rd:.3%} < Rf → Rf floor {self.rf:.3%}")
            return float(self.rf)
        self.report.add("rd", "ok", n_obs=int(valid.sum()), value=rd)
        return rd

    def compute_wacc(self, re, tax):
        """WACC = Re × E/V + Rd × (1-t) × D/V"""
        rd = self.compute_cost_of_debt(tax)

        # E (시가총액)
        mc, mc_date = load_korea_marketcap_latest(
            self.ticker_dg, self.db_info, table_name=TABLE_MARKETCAP)
        if mc is None or mc <= 0:
            # fallback: 현재가 × (자사주차감) 평균발행주식수
            wide = self._fs_wide
            cp = load_current_price(self.ticker_dg, self.db_info, table_name=TABLE_PRICE)
            shares_ser = wide.get("shares_treasury_adj", pd.Series([np.nan])).dropna()
            shares = shares_ser.iloc[-1] if not shares_ser.empty else np.nan
            if cp and not np.isnan(shares) and shares > 0:
                mkt_cap_won = cp * shares
                self.report.warn(f"시가총액 DB 없음 → 현재가×주식수={mkt_cap_won/1e12:.2f}조원 fallback")
            else:
                self.report.add("market_cap", "missing", n_obs=0,
                                note="시가총액·주가·주식수 모두 없음")
                if self.verbose: log(self.ticker_dg, "WACC: E 측정 실패 → Re만 사용")
                return float(re), re, rd  # WACC 대신 Re
        else:
            mkt_cap_won = mc * MARKETCAP_UNIT_MULTIPLIER  # 백만원 → 원

        # D (총부채)
        wide = self._fs_wide
        total_debt = pd.Series(0.0, index=wide.index)
        for k in DEBT_KEYS:
            if k in wide.columns:
                total_debt = total_debt + wide[k].fillna(0)
        if total_debt.dropna().empty:
            d_won = 0.0
        else:
            d_won = float(total_debt.sort_index().iloc[-1])

        V = mkt_cap_won + d_won
        if V <= 0:
            self.report.add("wacc_components", "missing", value=0.0)
            return float(re), re, rd

        we = mkt_cap_won / V
        wd = d_won / V
        wacc = re * we + rd * (1 - tax) * wd
        # ★ v8: WACC 하한 = max(절대하한, Rf+1%) — 절대 floor 가 고레버리지 기업
        #        WACC 를 Rf 이하로 떨어뜨려 TV 발산을 유발하던 문제 차단
        wacc_lo = max(V8_WACC_FLOOR_ABS, self.rf + 0.01)
        wacc = float(np.clip(wacc, wacc_lo, WACC_CAP))

        self._mkt_cap = mkt_cap_won
        if self.verbose:
            log(self.ticker_dg,
                f"WACC={wacc:.4%}  Re={re:.4%} Rd={rd:.4%} tax={tax:.4%}  "
                f"E/V={we:.2%} D/V={wd:.2%}  E={mkt_cap_won/1e12:.2f}조 D={d_won/1e12:.2f}조")
        return wacc, re, rd

    # ─────────────────────────────────────────────────────────
    # 5. EVA / Moat / Phase 2 (★ v9.2 패치 적용)
    # ─────────────────────────────────────────────────────────

    def _compute_eva_spread(self, wacc):
        """ROIC_TTM - WACC. (US v7 _compute_eva_spread 한국 어댑테이션)"""
        wide = self._fs_wide
        if "operating_income" not in wide.columns or "total_equity" not in wide.columns:
            return {"roic": np.nan, "eva_spread": np.nan, "ic": np.nan,
                    "n_positive": 0, "eva_series": []}

        tax = self.estimate_tax_rate()
        df = wide.copy()
        df["nopat_q"] = df["operating_income"] * (1 - tax)

        # IC = Equity + Total Debt - Cash
        td = pd.Series(0.0, index=df.index)
        for k in DEBT_KEYS:
            if k in df.columns:
                td = td + df[k].fillna(0)
        cash = pd.Series(0.0, index=df.index)
        for k in CASH_KEYS:
            if k in df.columns:
                cash = cash + df[k].fillna(0)
        df["ic"] = df["total_equity"].fillna(0) + td - cash

        df = df[["nopat_q", "ic"]].dropna().sort_index()
        if len(df) < 4:
            return {"roic": np.nan, "eva_spread": np.nan, "ic": np.nan,
                    "n_positive": 0, "eva_series": []}

        eva_series = []
        dates = sorted(df.index)
        for i, dt in enumerate(dates):
            if i < 3: continue
            nopat_ttm = float(df["nopat_q"].iloc[i-3:i+1].sum())
            ic_snap = float(df["ic"].iloc[i])
            if ic_snap <= 0: continue
            roic_q = nopat_ttm / ic_snap
            eva_q = roic_q - wacc
            eva_series.append({"date": dt, "roic": roic_q,
                               "eva_spread": eva_q, "ic": ic_snap})

        if not eva_series:
            return {"roic": np.nan, "eva_spread": np.nan, "ic": np.nan,
                    "n_positive": 0, "eva_series": []}
        latest = eva_series[-1]
        recent_20 = eva_series[-20:]
        n_positive = sum(1 for e in recent_20 if e["eva_spread"] > 0)

        if self.verbose:
            log(self.ticker_dg,
                f"EVA: ROIC={latest['roic']:.2%}  WACC={wacc:.2%}  "
                f"spread={latest['eva_spread']:+.2%}  "
                f"n_pos(20Q)={n_positive}/20")

        return {
            "roic": latest["roic"], "eva_spread": latest["eva_spread"],
            "ic": latest["ic"], "n_positive": n_positive,
            "eva_series": eva_series,
        }

    def _moat_to_rho_and_years(self, eva):
        """
        EVA spread → (ρ, Phase2 기간, label).

        등급 기준 (US v11 동일, 한국에도 동일하게 적용):
          Wide moat   : spread > 15% AND 최근 20Q 중 15Q+ 양수 → ρ=0.90, 15년
          Narrow moat : spread  8~15% AND 12Q+ 양수            → ρ=0.83, 12년
          Some moat   : spread  3~8%  AND  8Q+ 양수            → ρ=0.75,  8년
          No moat     : spread < 3%   OR   < 8Q 양수           → ρ=0.60,  5년

        근거: Mauboussin & Johnson (1997) FAJ — Competitive Advantage Period
              Damodaran (2002) Ch.12 — extraordinary growth Phase2 권고
              Chan, Karceski, Lakonishok (2003) JF — 성장 지속성 실증
        """
        if np.isnan(eva.get("eva_spread", np.nan)):
            return 0.75, 8, "Unknown (fallback)"
        s, n = eva["eva_spread"], eva["n_positive"]
        if   s > 0.15 and n >= 15: rho, yrs, lbl = 0.90, 15, "Wide moat"
        elif s > 0.08 and n >= 12: rho, yrs, lbl = 0.83, 12, "Narrow moat"
        elif s > 0.03 and n >=  8: rho, yrs, lbl = 0.75,  8, "Some moat"
        else:                       rho, yrs, lbl = 0.60,  5, "No moat"

        if self.verbose:
            log(self.ticker_dg,
                f"Moat: [{lbl}]  spread={s:+.2%}  n_pos={n}/20  "
                f"→ ρ={rho}  Phase2={yrs}yr")
        return rho, yrs, lbl

    # ═══════════════════════════════════════════════════════════════
    # ★ v9.2 패치: _estimate_phase2_growth — 다중 fallback + min(hist, fc)
    # ═══════════════════════════════════════════════════════════════

    def _estimate_phase2_growth(self, wacc):
        """
        Phase 2 AR(1) 성장 경로 — 음수 g₀ 허용 + 다중 fallback (★ v9.2).

        v1 → v2 변경:
          v1: 8Q-CAGR (양수 케이스만) → 실패시 forecast yr1/yr2,
              하한 G_TERM(0.025) 으로 음수 g₀ 거부.
          v2: 1단계 g0_hist (8Q-CAGR → 12Q-TTM 순) +
              2단계 g0_fc (forecast yr1/yr2, 항상 계산) +
              3단계 결합 로직:
                · 양수×양수: min(hist, fc) → 보수적 채택
                · forecast 음수: forecast 우선 → 사이클 침체 신호 신뢰
                · 부호 불일치: 더 보수적인 (작은) 값
              하한 G_FLOOR_MIN(-0.20) 으로 침체기 표현 가능

        반환: (g_path, moat_label, rho, n_years)
        """
        G_TERM = self.gdp_growth   # 한국 0.025

        h = self._fcff_history

        # ════════════════════════════════════════════════════════
        # 1단계: 과거 시계열 기반 g₀ (g0_hist)
        # ════════════════════════════════════════════════════════
        g0_hist = None
        g0_hist_method = None

        # ── Method 1: 8Q CAGR (양수 케이스만) ─────────────────────
        if h is not None and len(h.dropna()) >= 8:
            h_c = h.dropna()
            f0, fl = float(h_c.iloc[-8]), float(h_c.iloc[-1])
            if f0 > 0 and fl > 0:
                cagr_q = (fl / f0) ** (4.0 / 8) - 1
                g0_hist = float(np.clip((1 + cagr_q) ** 4 - 1, G_FLOOR_MIN, G_CEIL))
                g0_hist_method = "8Q-CAGR"

        # ── Method 2: 12Q rolling TTM vs TTM-2 (★ v9.2 신규) ────
        if g0_hist is None and h is not None and len(h.dropna()) >= 12:
            h_c = h.dropna()
            ttm_recent = float(h_c.iloc[-4:].sum())
            ttm_prior  = float(h_c.iloc[-8:-4].sum())
            if abs(ttm_prior) > 1e-6:
                if ttm_prior > 0 and ttm_recent > 0:
                    g0_hist = ttm_recent / ttm_prior - 1
                elif ttm_prior > 0 and ttm_recent <= 0:
                    g0_hist = ttm_recent / ttm_prior - 1   # 자연스럽게 음수
                elif ttm_prior < 0 and ttm_recent < 0:
                    g0_hist = (ttm_recent - ttm_prior) / abs(ttm_prior)
                else:                                       # 적자→흑자
                    g0_hist = G_TERM                        # 보수적
                g0_hist = float(np.clip(g0_hist, G_FLOOR_MIN, G_CEIL))
                g0_hist_method = "12Q-TTM"

        # ════════════════════════════════════════════════════════
        # 2단계: Phase 1 forecast 기반 g₀ (g0_fc) — 항상 계산 ★
        # ════════════════════════════════════════════════════════
        g0_fc = None
        g0_fc_method = None

        if self.result_df is not None and not self.result_df.empty:
            fcff_q = self.result_df["fcff"].values
            if len(fcff_q) >= 4:
                yr1 = float(np.sum(fcff_q[:4]))
                yr2 = float(np.sum(fcff_q[4:8])) if len(fcff_q) >= 8 else yr1
                if abs(yr1) > 1e-6:
                    if yr1 > 0 and yr2 > 0:
                        g0_fc = yr2 / yr1 - 1
                    elif yr1 > 0 and yr2 <= 0:
                        g0_fc = yr2 / yr1 - 1               # 자연스럽게 음수
                    elif yr1 < 0 and yr2 < 0:
                        g0_fc = (yr2 - yr1) / abs(yr1)
                    else:                                   # 적자→흑자
                        g0_fc = G_TERM
                    g0_fc = float(np.clip(g0_fc, G_FLOOR_MIN, G_CEIL))
                    g0_fc_method = "FC-yr1-yr2"

        # ════════════════════════════════════════════════════════
        # 3단계: g0_hist 와 g0_fc 결합 (★ v9.2 핵심 로직)
        # ════════════════════════════════════════════════════════
        if g0_hist is not None and g0_fc is not None:
            # 두 값 모두 양수 → min 채택 (보수적)
            if g0_hist > 0 and g0_fc > 0:
                if g0_hist > g0_fc:
                    g0 = g0_fc
                    g0_method = f"min({g0_hist_method}={g0_hist*100:.1f}%, FC={g0_fc*100:.1f}%) → FC"
                else:
                    g0 = g0_hist
                    g0_method = f"min({g0_hist_method}={g0_hist*100:.1f}%, FC={g0_fc*100:.1f}%) → {g0_hist_method}"
            # forecast 가 음수 → forecast 우선
            elif g0_fc <= 0:
                g0 = g0_fc
                g0_method = f"FC-priority (FC={g0_fc*100:.1f}% ≤ 0, hist={g0_hist*100:.1f}%)"
            # forecast 양수, hist 음수 → 더 보수적인 (작은) 값
            else:
                g0 = min(g0_hist, g0_fc)
                g0_method = f"min(hist={g0_hist*100:.1f}%, FC={g0_fc*100:.1f}%) → 부호 불일치 보수"
        elif g0_hist is not None:
            g0 = g0_hist
            g0_method = g0_hist_method
        elif g0_fc is not None:
            g0 = g0_fc
            g0_method = g0_fc_method
        else:
            # 최후 fallback
            g0 = G_TERM
            g0_method = "default-G_TERM"

        # ── EVA → 해자 등급 → (ρ, 기간) ────────────────────────────
        eva = self._compute_eva_spread(wacc)
        rho, n_years, moat_label = self._moat_to_rho_and_years(eva)

        # ── ρ OLS 보정 (★ v9.2: 양수 조건 제거 — 음수 annual 도 포함) ──
        if h is not None and len(h.dropna()) >= 12:
            h_c = h.dropna()
            annual_vals = [float(h_c.iloc[y:y+4].sum())
                           for y in range(0, len(h_c) - len(h_c) % 4, 4)]
            if len(annual_vals) >= 4:
                g_hist = [(annual_vals[i] / annual_vals[i-1] - 1)
                          if abs(annual_vals[i-1]) > 1e-6 else 0.0
                          for i in range(1, len(annual_vals))]
                if len(g_hist) >= 3:
                    yt = np.array([g - G_TERM for g in g_hist[1:]])
                    xt = np.array([g - G_TERM for g in g_hist[:-1]])
                    if np.dot(xt, xt) > 1e-10:
                        rho_ols = float(np.clip(
                            np.dot(xt, yt) / np.dot(xt, xt),
                            0.40, 0.92,
                        ))
                        # EVA 기반 ρ 와 OLS ρ 의 가중 평균 (각 50%)
                        rho = float(np.clip(0.5 * rho + 0.5 * rho_ols, 0.40, 0.92))
                        if self.verbose:
                            log(self.ticker_dg,
                                f"ρ 보정: EVA기반={rho:.3f}  OLS={rho_ols:.3f}  "
                                f"→ 혼합={rho:.3f}")

        # ── ★ v8: g₀ 기반 ρ 적응형 상한 (고성장일수록 빠른 평균회귀) ──
        #     g0>30% → ρ≤0.75(hyper) / >15% → ρ≤0.85(high) / ≤15% → ρ≤0.92(normal)
        #     근거: 고성장은 지속성이 낮음 → 평균회귀 가속 (Chan/Karceski/Lakonishok 2003)
        rho_cap_applied = False
        if V8_RHO_ADAPTIVE_CAP:
            if   g0 > 0.30: rho_cap = 0.75
            elif g0 > 0.15: rho_cap = 0.85
            else:           rho_cap = 0.92
            if rho > rho_cap:
                if self.verbose:
                    log(self.ticker_dg,
                        f"[v8] ρ 적응형 상한: g0={g0:.1%}  ρ {rho:.3f}→{rho_cap:.3f}")
                rho = rho_cap
                rho_cap_applied = True

        # ── AR(1) 성장 경로 — clip 하한 G_FLOOR_MIN 으로 완화 ────
        g = g0
        g_path = []
        for _ in range(n_years):
            g = G_TERM + (g - G_TERM) * rho
            g_path.append(float(np.clip(g, G_FLOOR_MIN, G_CEIL)))

        if self.verbose:
            log(self.ticker_dg,
                f"Phase2 [{moat_label}] ρ={rho:.3f} {n_years}년  "
                f"g0={g0:.1%} ({g0_method}) → {g_path[0]:.1%} ... {g_path[-1]:.1%}")

        # EVA 정보 + g₀ 진단 정보 캐싱 (export / decomposition 셀에서 사용)
        self._eva_cache = {
            "eva_spread":   eva.get("eva_spread", np.nan),
            "roic":         eva.get("roic",        np.nan),
            "n_positive":   eva.get("n_positive",  0),
            "moat_label":   moat_label,
            "rho":          rho,
            "n_phase2":     n_years,
            "eva_series":   eva.get("eva_series",  []),
            # ★ v9.2: g₀ 진단 정보
            "g0":           g0,
            "g0_method":    g0_method,
            "g0_hist":      g0_hist,
            "g0_hist_method": g0_hist_method,
            "g0_fc":        g0_fc,
            "g0_fc_method": g0_fc_method,
            "rho_cap_applied": rho_cap_applied,   # ★ v8
        }
        return g_path, moat_label, rho, n_years

    def compute_terminal_growth(self, reinv, wacc):
        """g_terminal ∈ [0, GDP_GROWTH] — Phase 2 에서 이미 수렴 처리되므로 보수적."""
        fcff_cagr = 0.0
        h = self._fcff_history
        if h is not None and len(h) >= 4:
            h20 = h.dropna().tail(20)
            f0, fl = float(h20.iloc[0]), float(h20.iloc[-1])
            n = len(h20)
            if f0 > 0 and fl > 0 and n >= 4:
                fcff_cagr = (fl / f0) ** (4.0 / n) - 1.0
                fcff_cagr = float(np.clip(fcff_cagr, -0.10, 0.15))
        roic_med = (float(self.result_df["roic"].dropna().median())
                    if not self.result_df["roic"].dropna().empty else 0.06)
        g_roic = roic_med * reinv
        # ★ v8: 영구성장률 상한 = min(GDP, Rf) (Damodaran: g_terminal ≤ 무위험수익률)
        g_cap = min(self.gdp_growth, self.rf) if V8_TERMINAL_RF_CAP else self.gdp_growth
        g = float(np.clip(0.5 * fcff_cagr + 0.5 * g_roic, 0.0, g_cap))

        if self.verbose:
            log(self.ticker_dg,
                f"g_terminal={g:.4f}  FCFF_CAGR={fcff_cagr:.4f}  g_roic={g_roic:.4f}")
        return g

    # ─────────────────────────────────────────────────────────
    # 6. Valuation — 3-Stage DCF
    # ─────────────────────────────────────────────────────────

    def compute_valuation(self):
        re, beta_blume = self.compute_beta_re()
        tax = self.estimate_tax_rate()
        wacc, re_used, rd = self.compute_wacc(re, tax)

        df = self.result_df.copy()
        # reinvestment_rate median
        pos = df[df["nopat"] > 0]["reinvestment_rate"].dropna()
        reinv_med = float(self._winsorize(pos).median()) if not pos.empty else 0.3

        # Phase 1: 분기 → 연간
        fcff_q = df["fcff"].values
        ph1_annual = np.array([float(np.sum(fcff_q[yr:yr+4]))
                               for yr in range(0, len(fcff_q), 4)])
        T_ph1 = len(ph1_annual)

        # Phase 2: AR(1)  ★ v9.2
        ph2_growth, moat_lbl, rho, n_ph2 = self._estimate_phase2_growth(wacc)
        ph2_annual = []
        last = float(ph1_annual[-1])
        for g in ph2_growth:
            last = last * (1 + g)
            ph2_annual.append(last)
        ph2_annual = np.array(ph2_annual)

        # Phase 3: TV  (★ v8 보수화 — 최소 스프레드 2% + TV 배수 상한)
        g_term = self.compute_terminal_growth(reinv_med, wacc)
        # ★ v8: 최소 스프레드 2% (구 1% → TV배수 100배+ 발산 차단)
        if wacc - g_term < V8_MIN_TV_SPREAD:
            g_term = max(0.0, wacc - V8_MIN_TV_SPREAD)
        fcff_last_annual = float(ph2_annual[-1])
        tv = fcff_last_annual * (1 + g_term) / (wacc - g_term)
        # ★ v8: TV / 마지막 연간 FCFF 배수 상한 (양(+) FCFF 에만, 이중 안전장치)
        tv_capped = False
        tv_fcff_mult = np.nan
        if fcff_last_annual > 0:
            tv_fcff_mult = tv / fcff_last_annual
            if V8_TV_FCFF_MULT_CAP and tv > V8_TV_FCFF_MULT_CAP * fcff_last_annual:
                tv = V8_TV_FCFF_MULT_CAP * fcff_last_annual
                tv_fcff_mult = float(V8_TV_FCFF_MULT_CAP)
                tv_capped = True
                if self.verbose:
                    log(self.ticker_dg,
                        f"[v8] TV 배수 상한 {V8_TV_FCFF_MULT_CAP:.0f}× 적용 "
                        f"(spread={wacc - g_term:.2%})")

        # PV
        all_annual = np.concatenate([ph1_annual, ph2_annual])
        T = len(all_annual)
        pv_fcff = float(sum(all_annual[t] / (1 + wacc) ** (t+1) for t in range(T)))
        pv_tv = tv / (1 + wacc) ** T
        ev = pv_fcff + pv_tv

        # Net Debt (latest)
        wide = self._fs_wide
        td = pd.Series(0.0, index=wide.index)
        for k in DEBT_KEYS:
            if k in wide.columns:
                td = td + wide[k].fillna(0)
        cash = pd.Series(0.0, index=wide.index)
        for k in CASH_KEYS:
            if k in wide.columns:
                cash = cash + wide[k].fillna(0)
        net_debt_series = (td - cash).dropna()
        net_debt = float(net_debt_series.iloc[-1]) if not net_debt_series.empty else 0.0

        equity_val = ev - net_debt

        # ── ★ v8: Sanity Guard — 산출 지분가치 vs 실제 시총 괴리 검사 ──
        #     입력오류(주식수/단위/FCFF)로 지분가치가 시총의 수천~수만 배가 되는
        #     사례를 차단. ValueError → process_one_ticker_kr 에서 fail 기록.
        mkt_cap_actual = self._mkt_cap if self._mkt_cap else np.nan
        mc_ratio = (equity_val / mkt_cap_actual
                    if (mkt_cap_actual and np.isfinite(mkt_cap_actual)
                        and mkt_cap_actual > 0)
                    else np.nan)
        if V8_SANITY_GUARD and np.isfinite(mc_ratio):
            if mc_ratio > V8_SANITY_MC_RATIO or (0 < mc_ratio < 1.0 / V8_SANITY_MC_RATIO):
                raise ValueError(
                    f"[{self.ticker_dg}] v8 sanity guard: 산출 지분가치 "
                    f"{equity_val/1e12:,.2f}조 vs 실제 시총 {mkt_cap_actual/1e12:,.2f}조 "
                    f"= {mc_ratio:,.1f}배 (허용 ±{V8_SANITY_MC_RATIO:.0f}배) → 평가 제외. "
                    f"입력(주식수/FCFF/NWC) 점검 필요")

        # Shares (자사주차감 우선)
        shares = np.nan
        for col in ["shares_treasury_adj", "shares_common"]:
            if col in wide.columns:
                s_ser = wide[col].dropna()
                s_ser = s_ser[s_ser > 0]
                if not s_ser.empty:
                    shares = float(s_ser.iloc[-1]); break
        if np.isnan(shares):
            self.report.add("shares", "missing", n_obs=0,
                            note="주식수 컬럼 모두 결측")
            target_price = np.nan
        else:
            target_price = equity_val / shares

        # 현재가
        cp = load_current_price(self.ticker_dg, self.db_info, table_name=TABLE_PRICE)
        self._current_price = cp
        upside = (((target_price/cp) - 1) * 100
                  if (cp and not np.isnan(target_price) and cp > 0) else np.nan)

        tv_weight = pv_tv / ev * 100 if ev != 0 else np.nan

        self.valuation = {
            "ticker": self.ticker_dg,
            "wacc": wacc, "wacc_re": re_used, "wacc_rd": rd,
            "beta_raw": self._beta_info.get("beta_raw") if self._beta_info else np.nan,
            "beta_blume": beta_blume,
            "g_terminal": g_term, "reinvestment_rate": reinv_med,
            "pv_fcff": pv_fcff, "terminal_value": tv, "pv_tv": pv_tv,
            "tv_weight_pct": tv_weight,
            "enterprise_value": ev, "net_debt": net_debt,
            "equity_value": equity_val, "shares": shares,
            "target_price": target_price, "current_price": cp,
            "upside_pct": upside,
            # 단계별 진단용 (Cell 7 / Cell 8 에서 사용)
            "ph1_annual": ph1_annual.tolist(),
            "ph2_annual": ph2_annual.tolist(),
            "ph2_growth": ph2_growth,
            "all_annual": all_annual.tolist(),
            "t_total":    T,
            "fcff_last_annual": fcff_last_annual,
            "annual_fcff": all_annual.tolist(),  # 구버전 호환 키
            # EVA / Moat / g₀ 정보
            "moat_label":   moat_lbl,
            "eva_spread":   self._eva_cache.get("eva_spread", np.nan),
            "roic":         self._eva_cache.get("roic", np.nan),
            "moat_rho":     rho,
            "rho_cap_applied": self._eva_cache.get("rho_cap_applied", False),  # ★ v8
            "tv_fcff_mult": tv_fcff_mult,   # ★ v8
            "tv_capped":    tv_capped,      # ★ v8
            "mc_ratio":     mc_ratio,       # ★ v8
            "n_phase2":     n_ph2,
            "eva_series":   self._eva_cache.get("eva_series", []),
            "g0":           self._eva_cache.get("g0", np.nan),
            "g0_method":    self._eva_cache.get("g0_method", "?"),
            "g0_hist":      self._eva_cache.get("g0_hist"),
            "g0_fc":        self._eva_cache.get("g0_fc"),
            # 한국 메타데이터
            "nwc_method":        getattr(self, "_nwc_method", "?"),
            "nwc_to_sales":      getattr(self, "_nwc_to_sales", np.nan),
            "op_current_assets": getattr(self, "_op_ca_latest", np.nan),
            "op_current_liab":   getattr(self, "_op_cl_latest", np.nan),
            "revenue_quarters": len(self._sales_actual),
            "forecast_model":   self._used_model,
            "forecast_date":    self._forecast_date,
            "t_years":          T,
        }

        if self.verbose:
            tp_str = f"{target_price:,.0f}원" if not np.isnan(target_price) else "N/A"
            cp_str = f"{cp:,.0f}원" if cp else "N/A"
            up_str = f"{upside:+.1f}%" if not np.isnan(upside) else "N/A"
            log(self.ticker_dg,
                f"3-Stage  EV={ev/1e12:.2f}조  TP={tp_str}  CP={cp_str}  Up={up_str}  "
                f"TV비중={tv_weight:.1f}%  Moat={moat_lbl}")
        return self

    # ─────────────────────────────────────────────────────────
    # 7. DB 저장 (v1 동일)
    # ─────────────────────────────────────────────────────────

    def save_to_db(self, run_date=None):
        if self.result_df is None or self.valuation is None:
            return 0
        run_date = run_date or datetime.now().strftime("%Y-%m-%d")
        v = self.valuation

        rows = []
        for _, row in self.result_df.iterrows():
            rows.append({
                "date": run_date, "ticker": self.ticker_dg,
                "quarter": row.get("quarter"),
                "sales_forecast": row.get("sales_forecast"),
                "opm_forecast": row.get("opm_forecast"),
                "ebit": row.get("ebit"), "tax_rate": row.get("tax_rate"),
                "nopat": row.get("nopat"), "da": row.get("da"),
                "capex": row.get("capex"), "nwc": row.get("nwc"),
                "delta_nwc": row.get("delta_nwc"), "fcff": row.get("fcff"),
                "roic": row.get("roic"),
                "reinvestment_rate": v["reinvestment_rate"],
                "g_terminal": v["g_terminal"],
                "discount_rate": v["wacc"],
                "wacc_re": v["wacc_re"], "wacc_rd": v["wacc_rd"],
                "beta_raw": v.get("beta_raw"), "beta_blume": v["beta_blume"],
                "enterprise_value": v["enterprise_value"],
                "net_debt": v["net_debt"], "equity_value": v["equity_value"],
                "shares": v["shares"], "target_price": v["target_price"],
                "current_price": v["current_price"], "upside_pct": v["upside_pct"],
                "moat_label": v["moat_label"], "eva_spread": v["eva_spread"],
                "moat_rho": v.get("moat_rho"),
                "rho_cap_applied": int(bool(v.get("rho_cap_applied", 0))),
                "tv_weight_pct": v.get("tv_weight_pct"),
                "tv_fcff_mult": v.get("tv_fcff_mult"),
                "tv_capped": int(bool(v.get("tv_capped", 0))),
                "mc_ratio": v.get("mc_ratio"),
                "nwc_method": v.get("nwc_method"),
                "nwc_to_sales": v.get("nwc_to_sales"),
                "op_current_assets": v.get("op_current_assets"),
                "op_current_liab": v.get("op_current_liab"),
                "revenue_quarters": v["revenue_quarters"],
                "forecast_model": v["forecast_model"],
                "forecast_date": v["forecast_date"],
            })

        save_df = pd.DataFrame(rows).where(pd.notnull(pd.DataFrame(rows)), None)
        # NaN/Inf → None
        for col in save_df.select_dtypes(include=[float]).columns:
            save_df[col] = save_df[col].apply(
                lambda x: None if (x is None or pd.isna(x) or np.isinf(x)) else x)

        sql = f'''
            INSERT INTO `{TABLE_RESULT}`
            (date, ticker, quarter, sales_forecast, opm_forecast, ebit, tax_rate,
             nopat, da, capex, nwc, delta_nwc, fcff, roic, reinvestment_rate,
             g_terminal, discount_rate, wacc_re, wacc_rd,
             beta_raw, beta_blume,
             enterprise_value, net_debt, equity_value,
             shares, target_price, current_price, upside_pct,
             moat_label, eva_spread,
             moat_rho, rho_cap_applied, tv_weight_pct, tv_fcff_mult, tv_capped, mc_ratio,
             nwc_method, nwc_to_sales, op_current_assets, op_current_liab,
             revenue_quarters,
             forecast_model, forecast_date)
            VALUES
            (%(date)s,%(ticker)s,%(quarter)s,%(sales_forecast)s,%(opm_forecast)s,
             %(ebit)s,%(tax_rate)s,%(nopat)s,%(da)s,%(capex)s,%(nwc)s,
             %(delta_nwc)s,%(fcff)s,%(roic)s,%(reinvestment_rate)s,
             %(g_terminal)s,%(discount_rate)s,%(wacc_re)s,%(wacc_rd)s,
             %(beta_raw)s,%(beta_blume)s,
             %(enterprise_value)s,%(net_debt)s,%(equity_value)s,
             %(shares)s,%(target_price)s,%(current_price)s,%(upside_pct)s,
             %(moat_label)s,%(eva_spread)s,
             %(moat_rho)s,%(rho_cap_applied)s,%(tv_weight_pct)s,%(tv_fcff_mult)s,%(tv_capped)s,%(mc_ratio)s,
             %(nwc_method)s,%(nwc_to_sales)s,%(op_current_assets)s,%(op_current_liab)s,
             %(revenue_quarters)s,
             %(forecast_model)s,%(forecast_date)s)
            ON DUPLICATE KEY UPDATE
                fcff=VALUES(fcff), target_price=VALUES(target_price),
                upside_pct=VALUES(upside_pct),
                enterprise_value=VALUES(enterprise_value),
                discount_rate=VALUES(discount_rate),
                g_terminal=VALUES(g_terminal),
                moat_rho=VALUES(moat_rho), rho_cap_applied=VALUES(rho_cap_applied),
                tv_weight_pct=VALUES(tv_weight_pct), tv_fcff_mult=VALUES(tv_fcff_mult),
                tv_capped=VALUES(tv_capped), mc_ratio=VALUES(mc_ratio)
        '''
        conn = get_pymysql_conn(db_info)
        try:
            with conn.cursor() as cur:
                cur.executemany(sql, save_df.to_dict("records"))
            conn.commit()
        except Exception:
            conn.rollback(); raise
        finally:
            conn.close()
        return len(rows)

    # ─────────────────────────────────────────────────────────
    # 8. 시각화 (v1 4-패널 미러 — 한국 단위 적용)
    # ─────────────────────────────────────────────────────────

    def plot(self):
        if self.result_df is None:
            print("[WARN] result_df 없음"); return

        df = self.result_df.copy()
        v = self.valuation or {}
        h = self._fcff_history

        # 단위: 조원 (1e12)
        UNIT, UNIT_LBL = 1e12, "조원"

        fig, axes = plt.subplots(2, 2, figsize=(16, 10))
        fig.suptitle(f"{self.ticker_dg}  Korea FCFF DCF Valuation v2",
                     fontsize=14, fontweight="bold")

        # ① 과거+예측 FCFF
        ax = axes[0, 0]
        if h is not None and not h.empty:
            h_plot = h.dropna().tail(16)
            h_lbl = [f"{d.year}Q{d.quarter}" for d in h_plot.index]
            ax.bar(range(len(h_plot)), h_plot.values/UNIT,
                   color=["#2980b9" if v_>=0 else "#c0392b" for v_ in h_plot.values],
                   alpha=0.75, label="Historical FCFF", edgecolor="white")
            ax.axvline(len(h_plot)-0.5, color="gray", lw=1.2, ls="--",
                       label="Forecast start")
            xt = list(range(len(h_plot))); xl = h_lbl; off = len(h_plot)
        else:
            xt, xl, off = [], [], 0
        fc_col = ["#27ae60" if f>=0 else "#e74c3c" for f in df["fcff"]]
        ax.bar(range(off, off+len(df)), df["fcff"].values/UNIT,
               color=fc_col, alpha=0.90, label="Forecast FCFF", edgecolor="white")
        xt += list(range(off, off+len(df))); xl += df["quarter"].tolist()
        ax.axhline(0, color="black", lw=0.8)
        ax.set_xticks(xt); ax.set_xticklabels(xl, rotation=45, ha="right", fontsize=7)
        ax.set_title(f"Historical & Forecast FCFF ({UNIT_LBL})")
        ax.set_ylabel(UNIT_LBL); ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)

        # ② Sales + EBIT
        ax = axes[0, 1]
        act = self._sales_actual
        if act is not None and not act.empty:
            ap = act.tail(16); al = [f"{d.year}Q{d.quarter}" for d in ap.index]
            ax.plot(range(len(ap)), ap.values/UNIT, marker="o", ms=3, lw=1.5,
                    color="#2980b9", label="Actual Sales", zorder=3)
            ax.axvline(len(ap)-0.5, color="gray", lw=1.2, ls="--")
            so, sx, sl = len(ap), list(range(len(ap))), al
        else:
            so, sx, sl = 0, [], []
        ax.bar(range(so, so+len(df)), df["sales_forecast"].values/UNIT,
               alpha=0.45, label="Forecast Sales", color="#3498db")
        ax.bar(range(so, so+len(df)), df["ebit"].values/UNIT,
               alpha=0.70, label="Forecast EBIT", color="#e67e22")
        sx += list(range(so, so+len(df))); sl += df["quarter"].tolist()
        ax.set_xticks(sx); ax.set_xticklabels(sl, rotation=45, ha="right", fontsize=7)
        ax.set_title(f"Actual Sales → Forecast Sales & EBIT ({UNIT_LBL})")
        ax.set_ylabel(UNIT_LBL); ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)

        # ③ OPM
        ax = axes[1, 0]
        wide = self._fs_wide
        if "operating_income" in wide.columns and "revenue" in wide.columns:
            opm_h = (wide["operating_income"]/wide["revenue"]).dropna().tail(16)
            ax.plot(range(len(opm_h)), opm_h.values*100, marker="o", ms=3, lw=1.5,
                    color="purple", label="Historical OPM")
            ax.axvline(len(opm_h)-0.5, color="gray", lw=1.2, ls="--")
            oo, ox, oxl = (len(opm_h), list(range(len(opm_h))),
                           [f"{d.year}Q{d.quarter}" for d in opm_h.index])
        else:
            oo, ox, oxl = 0, [], []
        ax.plot(range(oo, oo+len(df)), df["opm_forecast"].values*100,
                marker="s", ms=3, lw=1.5, ls="--", color="#8e44ad",
                label="Forecast OPM")
        ox += list(range(oo, oo+len(df))); oxl += df["quarter"].tolist()
        ax.set_xticks(ox); ax.set_xticklabels(oxl, rotation=45, ha="right", fontsize=7)
        ax.axhline(0, color="black", lw=0.8); ax.set_title("OPM (%)")
        ax.set_ylabel("%"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_:f"{x:.1f}%"))

        # ④ 요약
        ax = axes[1, 1]; ax.axis("off")
        def _fmt(x, unit="조", d=2):
            if pd.isna(x) or (isinstance(x,float) and np.isnan(x)): return "N/A"
            if unit == "조": return f"{x/1e12:,.{d}f}조원"
            if unit == "%":  return f"{x*100:.{d}f}%"
            return f"{x:,.{d}f}원"
        lines = [
            f"Current Price : {_fmt(v.get('current_price'),'원',0)}",
            f"Target Price  : {_fmt(v.get('target_price'),'원',0)}",
            f"Upside        : {v.get('upside_pct',0):+.1f}%" if not np.isnan(v.get('upside_pct',np.nan)) else "Upside : N/A",
            "─" * 32,
            f"PV(FCFF)      : {_fmt(v.get('pv_fcff'))}",
            f"Terminal Val  : {_fmt(v.get('terminal_value'))}",
            f"Enterprise V  : {_fmt(v.get('enterprise_value'))}",
            f"Net Debt      : {_fmt(v.get('net_debt'))}",
            f"Equity Value  : {_fmt(v.get('equity_value'))}",
            "─" * 32,
            f"WACC          : {_fmt(v.get('wacc'),'%')}",
            f"  Re          : {_fmt(v.get('wacc_re'),'%')}",
            f"  Rd          : {_fmt(v.get('wacc_rd'),'%')}",
            f"  β_blume     : {v.get('beta_blume',np.nan):.3f}",
            f"g_terminal    : {_fmt(v.get('g_terminal'),'%')}",
            "─" * 32,
            f"Moat          : {v.get('moat_label','N/A')}",
            f"EVA spread    : {_fmt(v.get('eva_spread'),'%')}",
            f"g₀ method     : {v.get('g0_method','?')}",
            f"NWC 정의      : {v.get('nwc_method','?')}",
            f"Quarters used : {v.get('revenue_quarters',0)}",
            f"Forecast model: {v.get('forecast_model','N/A')}",
        ]
        ax.text(0.05, 0.97, "\n".join(lines), transform=ax.transAxes,
                fontsize=9, verticalalignment="top", fontfamily="monospace",
                bbox=dict(boxstyle="round,pad=0.5", facecolor="#f8f9fa", alpha=0.8))
        plt.tight_layout(); plt.show()

    # ─────────────────────────────────────────────────────────
    # 9. 전체 실행
    # ─────────────────────────────────────────────────────────

    def run(self):
        self.load_sales()
        self.load_financials()
        self.compute_fcff()
        self.compute_valuation()
        return self


# ── 단일 티커 처리 래퍼 ──────────────────────────────────────────
def process_one_ticker_kr(
    ticker, engine, db_info, rf, e_rm, kospi_series,
    verbose=False, run_date=None, save_db=True,
):
    """KoreaDCFModel 실행 + 결과 + 데이터 품질 리포트 반환."""
    run_date = run_date or datetime.now().strftime("%Y-%m-%d")
    try:
        m = KoreaDCFModel(ticker=ticker, engine=engine, db_info=db_info,
                          rf=rf, e_rm=e_rm, kospi_series=kospi_series,
                          verbose=verbose)
        m.run()
        rows = m.save_to_db(run_date) if save_db else 0
        v = m.valuation
        return {
            "status": "ok", "ticker": m.ticker_dg,
            "target_price":  v.get("target_price", np.nan),
            "current_price": v.get("current_price", np.nan),
            "upside_pct":    v.get("upside_pct", np.nan),
            "wacc":          v.get("wacc", np.nan),
            "g_terminal":    v.get("g_terminal", np.nan),
            "moat":          v.get("moat_label", "?"),
            "rows_saved":    rows,
            "report":        m.report,
            "msg": (f"TP={v.get('target_price',np.nan):,.0f}원"
                    if not np.isnan(v.get('target_price', np.nan)) else "TP=N/A"),
        }
    except Exception as e:
        return {
            "status": "fail", "ticker": to_dg_ticker(ticker),
            "target_price": np.nan, "current_price": np.nan,
            "upside_pct": np.nan, "wacc": np.nan, "g_terminal": np.nan,
            "moat": "?", "rows_saved": 0,
            "report": None,
            "msg": str(e)[:120],
        }
    finally:
        clear_memory()


print("[OK] KoreaDCFModel v8 클래스 & process_one_ticker_kr 정의 완료")


[OK] KoreaDCFModel v8 클래스 & process_one_ticker_kr 정의 완료


## Cell 8 · 배치 실행 → DB 저장

- 체크포인트: 완료/실패 종목을 `_korea_fcff_checkpoint/` 에 기록 (`SKIP_DONE=True` 로 이어하기)
- 종료 후 데이터 품질 리포트를 `TABLE_QUALITY` 에 저장

In [8]:
# ─────────────────────────────────────────────────────────────
#  Cell 10 · 배치 실행 (v2)
#  - v1 동작 그대로 유지
#  - process_one_ticker_kr() 내부에서 v8/v9.2 패치된 KoreaDCFModel 사용
# ─────────────────────────────────────────────────────────────

# ★ 입력변수 셀(Cell 2)의 RUN_TICKERS_OVERRIDE 가 비어있지 않으면 그 리스트만 실행
if RUN_TICKERS_OVERRIDE:
    RUN_TICKERS = [to_dg_ticker(t) for t in RUN_TICKERS_OVERRIDE]
    log("BATCH", f"OVERRIDE 모드: 지정 종목 {len(RUN_TICKERS)}개만 실행")
else:
    RUN_TICKERS = KOREA_TICKER_LIST[TICKER_START:TICKER_END]
total = len(RUN_TICKERS)
run_date = datetime.now().strftime("%Y-%m-%d")

done_set = set()
if SKIP_DONE and os.path.exists(DONE_PATH):
    with open(DONE_PATH, encoding="utf-8") as f:
        done_set = {l.strip() for l in f if l.strip()}

ok_cnt = skip_cnt = fail_cnt = 0
results = []
all_reports = []
t0 = time.time()

log("BATCH", f"배치 시작 (v2): {total:,}개  run_date={run_date}  SKIP_DONE={SKIP_DONE}")
print("=" * 80)

for idx, ticker in enumerate(RUN_TICKERS, 1):
    pct = idx / total * 100
    prefix = f"[{idx:>5}/{total}] ({pct:5.1f}%) {ticker:<8}"

    if SKIP_DONE and ticker in done_set:
        print(f"{prefix} SKIP", flush=True); skip_cnt += 1; continue

    res = process_one_ticker_kr(
        ticker, engine, db_info, RF, E_RM, KOSPI_PX,
        verbose=False, run_date=run_date, save_db=True,
    )
    results.append(res)
    if res["report"] is not None:
        all_reports.append(res["report"])

    if res["status"] == "ok":
        tp = res["target_price"]; up = res["upside_pct"]; w = res["wacc"]
        tp_s = f"TP={tp:,.0f}원" if not np.isnan(tp) else "TP=N/A"
        up_s = f"↑{up:+.1f}%" if not np.isnan(up) else ""
        print(f"{prefix} OK  {tp_s} {up_s}  WACC={w:.3%}  Moat={res['moat']}", flush=True)
        with open(DONE_PATH, "a", encoding="utf-8") as f:
            f.write(ticker + "\n")
        ok_cnt += 1
    else:
        print(f"{prefix} FAIL  {res['msg']}", flush=True)
        with open(FAIL_PATH, "a", encoding="utf-8") as f:
            f.write(f"{ticker}\t{res['msg']}\n")
        fail_cnt += 1

elapsed = time.time() - t0
print("=" * 80)
log("BATCH", f"완료  OK={ok_cnt}  SKIP={skip_cnt}  FAIL={fail_cnt}  "
             f"경과={elapsed:.0f}s  평균={elapsed/max(ok_cnt+fail_cnt,1):.1f}s/ticker")

# ── 데이터 품질 리포트 DB 저장 ────────────────────────────────
if all_reports:
    n_quality = save_quality_report_to_db(
        all_reports, db_info, table_name=TABLE_QUALITY,
        run_date=run_date, model_name="FCFF_DCF",
    )
    log("QUALITY", f"데이터 품질 로그 저장: {n_quality:,}건 → {TABLE_QUALITY}")

# ── 결과 요약 ────────────────────────────────────────────────
if results:
    summary = pd.DataFrame([{
        "ticker": r["ticker"], "target_price": r["target_price"],
        "current_price": r["current_price"], "upside_pct": r["upside_pct"],
        "wacc": r["wacc"], "g_terminal": r["g_terminal"], "moat": r["moat"],
        "status": r["status"],
    } for r in results])
    ok_summary = summary[summary["status"] == "ok"].sort_values(
        "upside_pct", ascending=False)
    print("\n[상위 업사이드 종목 TOP 20]")
    display(ok_summary.head(20))


[15:44:27][BATCH] 배치 시작 (v2): 1,289개  run_date=2026-08-27  SKIP_DONE=False
[메모리] forecast_sarima 실행 전: 482.83 MB
[메모리] find_best_sarima_params 실행 전: 482.85 MB
[메모리] find_best_sarima_params 실행 후: 486.00 MB (변화: +3.15 MB)
[메모리] forecast_sarima 실행 후: 486.04 MB (변화: +3.20 MB)
[메모리] forecast_ets 실행 전: 486.04 MB
[메모리] forecast_ets 실행 후: 486.30 MB (변화: +0.26 MB)
[메모리] forecast_theta 실행 전: 486.30 MB
[메모리] forecast_theta 실행 후: 486.45 MB (변화: +0.16 MB)
[    1/1289] (  0.1%) A017800  OK  TP=93,093원 ↑+36.9%  WACC=9.875%  Moat=No moat
[메모리] forecast_sarima 실행 전: 486.65 MB
[메모리] find_best_sarima_params 실행 전: 486.65 MB
[메모리] find_best_sarima_params 실행 후: 486.65 MB (변화: +0.00 MB)
[메모리] forecast_sarima 실행 후: 486.65 MB (변화: +0.00 MB)
[메모리] forecast_ets 실행 전: 486.65 MB
[메모리] forecast_ets 실행 후: 486.66 MB (변화: +0.01 MB)
[메모리] forecast_theta 실행 전: 486.66 MB
[메모리] forecast_theta 실행 후: 486.66 MB (변화: +0.00 MB)
[    2/1289] (  0.2%) A017900  OK  TP=2,195원 ↑-57.1%  WACC=11.132%  Moat=No moat
[메모리] forecast_sari

,ticker,target_price,current_price,upside_pct,wacc,g_terminal,moat,status
374,A009200,4.162848e+04,1427.0,2817.202526,0.092084,0.04,No moat,ok
143,A017940,2.274721e+06,87000.0,2514.622323,0.087744,0.04,No moat,ok
436,A084690,1.791426e+05,7300.0,2354.008346,0.091378,0.04,No moat,ok
40,A095570,1.008065e+05,4200.0,2300.154029,0.087750,0.04,No moat,ok
425,A058860,5.683238e+04,2470.0,2200.906175,0.090866,0.04,No moat,ok
17,A044450,2.165538e+05,9630.0,2148.741094,0.084054,0.04,No moat,ok
851,A038390,2.112418e+05,9660.0,2086.768593,0.084727,0.04,No moat,ok
196,A002320,3.180187e+05,14660.0,2069.295568,0.089767,0.04,No moat,ok
259,A013580,3.765817e+05,17830.0,2012.067699,0.101201,0.04,No moat,ok
146,A021050,2.023068e+04,969.0,1987.789102,0.102086,0.04,No moat,ok


## Cell 9 · 결과 조회 & 시각화 (DB 현황·업사이드 분포)

In [ ]:
# ─────────────────────────────────────────────────────────────
#  Cell 11 · 결과 조회 & 시각화 (v2 — v1 동일)
# ─────────────────────────────────────────────────────────────

# ── 8-1. DB 현황 (날짜별 평가 기록) ─────────────────────────
conn = get_pymysql_conn(db_info)
try:
    with conn.cursor() as cur:
        cur.execute(f"""
            SELECT DATE(date) AS run_date,
                   COUNT(DISTINCT ticker) AS tickers,
                   COUNT(*) AS total_rows,
                   AVG(target_price) AS avg_tp,
                   AVG(upside_pct) AS avg_upside,
                   AVG(discount_rate) AS avg_wacc
            FROM {TABLE_RESULT}
            GROUP BY DATE(date)
            ORDER BY run_date DESC LIMIT 10
        """)
        _summary = pd.DataFrame(cur.fetchall())
finally:
    conn.close()

print("=" * 70)
print(f"[DB 현황] {TABLE_RESULT}")
print("=" * 70)
display(_summary)

# ── 8-2. 최신 평가일 기준 업사이드 분포 ─────────────────────
conn = get_pymysql_conn(db_info)
try:
    with conn.cursor() as cur:
        cur.execute(f"SELECT MAX(date) AS m FROM {TABLE_RESULT}")
        max_date = cur.fetchone()["m"]
        if max_date:
            cur.execute(f"""
                SELECT ticker,
                       MAX(target_price) AS target_price,
                       MAX(current_price) AS current_price,
                       MAX(upside_pct) AS upside_pct,
                       MAX(discount_rate) AS wacc,
                       MAX(g_terminal) AS g_terminal,
                       MAX(moat_label) AS moat
                FROM {TABLE_RESULT}
                WHERE date=%s AND target_price IS NOT NULL AND current_price IS NOT NULL
                GROUP BY ticker
                ORDER BY upside_pct DESC
            """, (max_date,))
            results_df = pd.DataFrame(cur.fetchall())
        else:
            results_df = pd.DataFrame()
finally:
    conn.close()

if max_date and not results_df.empty:
    print(f"\n[최신 평가일: {max_date}] 총 {len(results_df)}개")
    print("\n🔼 업사이드 TOP 20")
    display(results_df.head(20))
    print("\n🔽 다운사이드 TOP 20")
    display(results_df.tail(20))

    # 분포 시각화
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"Korea Valuation Summary v2 ({max_date})", fontsize=13)
    ax = axes[0]
    upside_clean = results_df["upside_pct"].dropna().clip(-200, 200)
    ax.hist(upside_clean, bins=40, color="#3498db", edgecolor="white", alpha=0.8)
    ax.axvline(0, color="black", lw=1)
    ax.axvline(20, color="green", lw=1, ls="--", label="+20% (Buy zone)")
    ax.set_title("Upside Distribution")
    ax.set_xlabel("Upside (%)"); ax.legend(); ax.grid(alpha=0.3)

    ax = axes[1]
    grade = pd.cut(results_df["upside_pct"],
                   bins=[-np.inf, -20, 0, 20, 50, np.inf],
                   labels=["Strong Sell", "Sell", "Hold", "Buy", "Strong Buy"])
    grade.value_counts().sort_index().plot(
        kind="barh", ax=ax,
        color=["#c0392b", "#e74c3c", "#f39c12", "#2ecc71", "#27ae60"])
    ax.set_title("Valuation Grade"); ax.grid(axis="x", alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print("[INFO] 데이터 없음 — Cell 10 배치 실행 필요")

# ── 8-3. 종목별 평가일자 추이 ───────────────────────────────
print("\n[종목별 평가 이력 예시 — 삼성전자]")
hist = get_evaluation_history("A005930", db_info, table_name=TABLE_RESULT)
display(hist)


## Cell 10 · 데이터 품질 진단

1. 항목별 status 분포 (`ok / fallback_median / fallback_zero / missing`)
2. fallback 비율 TOP 항목
3. 문제 종목 TOP 30 — `inspect_ticker_quality("A005930")` 로 종목별 상세 조회 가능

In [ ]:
# ─────────────────────────────────────────────────────────────
#  Cell 12 · 데이터 품질 진단 (v2 — v1 동일)
# ─────────────────────────────────────────────────────────────

conn = get_pymysql_conn(db_info)
try:
    with conn.cursor() as cur:
        # 1) 최신 평가일의 status별 분포
        cur.execute(f"""
            SELECT field, status, COUNT(*) AS cnt,
                   AVG(r_squared) AS avg_r2,
                   AVG(value) AS avg_value
            FROM {TABLE_QUALITY}
            WHERE date = (SELECT MAX(date) FROM {TABLE_QUALITY} WHERE model='FCFF_DCF')
              AND model='FCFF_DCF'
            GROUP BY field, status
            ORDER BY field, status
        """)
        df_status = pd.DataFrame(cur.fetchall())

        # 2) fallback 비율 높은 항목 TOP 10
        cur.execute(f"""
            SELECT field,
                   SUM(status='ok') AS ok_cnt,
                   SUM(status='fallback_median') AS fb_med,
                   SUM(status='fallback_zero') AS fb_zero,
                   SUM(status='missing') AS miss,
                   COUNT(*) AS total,
                   ROUND(SUM(status LIKE 'fallback%')/COUNT(*)*100, 1) AS fb_pct
            FROM {TABLE_QUALITY}
            WHERE date = (SELECT MAX(date) FROM {TABLE_QUALITY} WHERE model='FCFF_DCF')
              AND model='FCFF_DCF'
            GROUP BY field
            ORDER BY fb_pct DESC
        """)
        df_field = pd.DataFrame(cur.fetchall())

        # 3) 가장 문제 많은 종목 TOP 30
        cur.execute(f"""
            SELECT ticker,
                   SUM(status LIKE 'fallback%') AS fallback_cnt,
                   SUM(status='missing') AS miss_cnt,
                   GROUP_CONCAT(DISTINCT field ORDER BY field SEPARATOR ',') AS issues
            FROM {TABLE_QUALITY}
            WHERE date = (SELECT MAX(date) FROM {TABLE_QUALITY} WHERE model='FCFF_DCF')
              AND model='FCFF_DCF'
              AND (status LIKE 'fallback%' OR status='missing')
            GROUP BY ticker
            HAVING fallback_cnt + miss_cnt >= 3
            ORDER BY (fallback_cnt + miss_cnt*2) DESC
            LIMIT 30
        """)
        df_problem_tickers = pd.DataFrame(cur.fetchall())
finally:
    conn.close()

print("=" * 70); print("[1] 항목별 status 분포 (최신 평가일)"); print("=" * 70)
display(df_status)

print("\n" + "=" * 70); print("[2] 항목별 fallback 비율 (TOP)"); print("=" * 70)
display(df_field)

print("\n" + "=" * 70); print("[3] 데이터 문제가 많은 종목 TOP 30 (수동 확인 권장)"); print("=" * 70)
display(df_problem_tickers)

# ── 4) 특정 종목의 모든 진단 항목 조회 함수 ──────────────
def inspect_ticker_quality(ticker: str, run_date: str = None) -> pd.DataFrame:
    """특정 종목의 데이터 품질 상세 (해당 평가일 또는 최신)."""
    tk = to_dg_ticker(ticker)
    conn = get_pymysql_conn(db_info)
    try:
        with conn.cursor() as cur:
            if run_date:
                cur.execute(f"""SELECT date, field, status, n_obs, value, r_squared, note
                                  FROM {TABLE_QUALITY}
                                  WHERE ticker=%s AND date=%s AND model='FCFF_DCF'
                                  ORDER BY field""", (tk, run_date))
            else:
                cur.execute(f"""SELECT date, field, status, n_obs, value, r_squared, note
                                  FROM {TABLE_QUALITY}
                                  WHERE ticker=%s AND model='FCFF_DCF'
                                    AND date = (SELECT MAX(date) FROM {TABLE_QUALITY}
                                                 WHERE ticker=%s AND model='FCFF_DCF')
                                  ORDER BY field""", (tk, tk))
            return pd.DataFrame(cur.fetchall())
    finally:
        conn.close()

# 사용 예시
print("\n[예시] 삼성전자 데이터 품질 상세")
display(inspect_ticker_quality("A005930"))


## Cell 11 · Upside 랭킹 & 요구수익률/WACC 테이블

배치 결과에서 지정 측정일(`RANK_DATES_INPUT`, Cell 2) 기준 종목별 최신 1행으로 집계 후 upside 상위 `TOP_N` 종목 추출 → CSV 저장

In [ ]:
# ─────────────────────────────────────────────────────────────
#  Cell 13 (★ v7) · 측정일자 조회 + 날짜 리스트 기반 Upside 랭킹
#  - 입력한 날짜들에 측정된 기업의 valuation 출력
#  - 같은 기업이 여러 날 측정됐으면 '가장 최근 측정일' 행만 사용
# ─────────────────────────────────────────────────────────────

# ═══════════════════════════════════════════════════════════════
#  13-0. 측정된 날짜 목록 조회 (어떤 날에 몇 종목을 측정했는지)
# ═══════════════════════════════════════════════════════════════
sql_dates = text(f"""
    SELECT `date` AS run_date, COUNT(DISTINCT ticker) AS n_tickers
    FROM `{TABLE_RESULT}`
    GROUP BY `date`
    ORDER BY `date` DESC
""")
dates_df = pd.read_sql(sql_dates, engine)
dates_df["run_date"] = pd.to_datetime(dates_df["run_date"]).dt.strftime("%Y-%m-%d")

print(f"\n{'='*60}")
print(f"  측정된 날짜 목록 (총 {len(dates_df)}일)")
print(f"{'='*60}")
display(dates_df)

# ═══════════════════════════════════════════════════════════════
#  13-1. 조회할 날짜 리스트 — Cell 2 입력변수 셀의 RANK_DATES_INPUT 에서 지정
#       - RANK_DATES = []        → 가장 최근 측정일 1일만 자동 사용
#       - RANK_DATES = ["2026-06-01", "2026-04-21"]  → 지정한 날짜들
# ═══════════════════════════════════════════════════════════════
RANK_DATES = list(RANK_DATES_INPUT)   # ★ Cell 2 입력변수 셀의 RANK_DATES_INPUT 사용

if not RANK_DATES:
    RANK_DATES = [dates_df["run_date"].iloc[0]]   # 최신 1일
    log("RANK", f"날짜 미지정 → 최신 측정일 사용: {RANK_DATES}")
else:
    # 입력값 정규화 (datetime/Timestamp 가 들어와도 'YYYY-MM-DD' 로)
    RANK_DATES = [pd.to_datetime(d).strftime("%Y-%m-%d") for d in RANK_DATES]
    valid = set(dates_df["run_date"])
    missing = [d for d in RANK_DATES if d not in valid]
    if missing:
        log("RANK", f"⚠️  측정 기록 없는 날짜 (무시됨): {missing}")
    RANK_DATES = [d for d in RANK_DATES if d in valid]
    if not RANK_DATES:
        raise ValueError("입력한 날짜 중 측정 기록이 있는 날짜가 하나도 없습니다.")

log("RANK", f"조회 날짜 {len(RANK_DATES)}일 = {sorted(RANK_DATES, reverse=True)}   TOP_N={TOP_N}")

# ═══════════════════════════════════════════════════════════════
#  13-2. 날짜 리스트 기반 집계
#    - 같은 ticker 가 여러 날 측정됐으면 가장 최근 date 행만 선택
#    - 윈도우 함수로 '종목별 최신 측정일 행 전체'를 통째로 추출
#      (MAX 컬럼별 집계와 달리 행 내 필드 정합성 유지)
# ═══════════════════════════════════════════════════════════════
_placeholders = ", ".join([f":d{i}" for i in range(len(RANK_DATES))])
_params = {f"d{i}": d for i, d in enumerate(RANK_DATES)}

sql_rank = text(f"""
    SELECT *
    FROM (
        SELECT
            ticker, `date` AS measured_date,
            target_price, current_price, upside_pct,
            beta_raw, beta_blume,
            wacc_re  AS cost_of_equity,
            wacc_rd  AS cost_of_debt,
            discount_rate AS wacc,
            g_terminal, moat_label AS moat, eva_spread,
            nwc_method, enterprise_value, equity_value,
            net_debt, revenue_quarters,
            ROW_NUMBER() OVER (
                PARTITION BY ticker
                ORDER BY `date` DESC, id DESC
            ) AS rn
        FROM `{TABLE_RESULT}`
        WHERE `date` IN ({_placeholders})
          AND target_price IS NOT NULL
          AND current_price > 0
          AND upside_pct IS NOT NULL
    ) t
    WHERE rn = 1
""")
rank_all = pd.read_sql(sql_rank, engine, params=_params)
rank_all = rank_all.drop(columns=["rn"])
rank_all["measured_date"] = pd.to_datetime(rank_all["measured_date"]).dt.strftime("%Y-%m-%d")
log("RANK", f"중복 제거 후 유효 종목 {len(rank_all):,}개")

# 3. upside 내림차순 → 상위 N
rank_top = (rank_all.sort_values("upside_pct", ascending=False)
                    .head(TOP_N)
                    .reset_index(drop=True))
rank_top.index = rank_top.index + 1
rank_top.index.name = "rank"

# 4. 표시용 포맷 테이블 (한글 라벨)
disp = pd.DataFrame({
    "ticker":               rank_top["ticker"],
    "측정일":               rank_top["measured_date"],
    "목표주가(원)":          rank_top["target_price"].round(0),
    "현재가격(원)":          rank_top["current_price"].round(0),
    "Upside(%)":            rank_top["upside_pct"].round(1),
    "β_raw":                rank_top["beta_raw"].round(3),
    "β_Blume":              rank_top["beta_blume"].round(3),
    "주주요구수익률 Re(%)":  (rank_top["cost_of_equity"] * 100).round(2),
    "부채요구수익률 Rd(%)":  (rank_top["cost_of_debt"]   * 100).round(2),
    "WACC(%)":              (rank_top["wacc"]            * 100).round(2),
    "g_term(%)":            (rank_top["g_terminal"]      * 100).round(2),
    "Moat":                 rank_top["moat"],
    "NWC정의":              rank_top["nwc_method"],
}, index=rank_top.index)

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print(f"\n{'='*86}")
print(f"  Upside 상위 {TOP_N} 종목   (측정일 {sorted(RANK_DATES, reverse=True)},  중복=최근측정 우선)")
print(f"{'='*86}")
display(disp)

# 5. CSV 저장
_date_tag = f"{min(RANK_DATES)}_to_{max(RANK_DATES)}" if len(RANK_DATES) > 1 else RANK_DATES[0]
out_csv = os.path.join(CHECKPOINT_DIR, f"fcff_upside_top{TOP_N}_{_date_tag}.csv")
disp.to_csv(out_csv, encoding="utf-8-sig")
log("RANK", f"CSV 저장: {out_csv}")